# Init

This is an example notebook that tries to show how to generate the gds of TWPAs with overlap junctions.
It uses the classes TWPA_elements, TWPA_assembly, TWPA_parameters and CAD_manager.

Up to now (30/01/2026), five different devices have been implemented:
1) Left-Handed TWPAs
2) Right-Handed TWPAs with parallel plate capacitors (with or without modulation)
3) Right-Handed TWPAs with top ground (with or without modulation)
4) Composite Right-Left-Handed TWPAs
5) Composite Right-Left-Handed TWPAs 2.0
6) SNAIL TWPAs with top ground (with or without modulation)
7) Lumped Element Resonators

The devices and fabrication parameters are defined in the json files fab_parameters and device_parameters. 
These parameters are passed to the class TWPA_parameters that returns design parameters (like dimensions of capacitors) depending on the input ones.

All the design parameters are then passed to the TWPA_elements class the generate the desired unit cell. This is used by the TWPA_assembly class to generate the device.

The generation of the gds and of the job and batch files passes through the CAD_manager class. 

In [1]:
import gdstk
import numpy as np
import os
import importlib
from pprint import pprint
import json

import sys

In [2]:
import TWPA_elements as TWPA_elements_cls
import CAD_manager as CAD_manager_cls
import TWPA_parameters as TWPA_parameters_cls

In [3]:
date = '260415'
WaferName = 'Feedline_test'
WaferNumber = '11'

In [4]:
importlib.reload(CAD_manager_cls)
CAD_manager = CAD_manager_cls.CAD_manager(date, WaferName, WaferNumber)

Junk folder already exists, cleaning it...
Junk folder already exists, cleaning it...


# Markers

In [5]:
CAD_manager.generate_markers()

# Litho and doses definition

Let's define the layers where we are going to put the structures on

In [6]:
ground_layer = 1
pad_bottom_layer = 2
connection_wire_bottom_layer = 3
jj_bottom_layer = 4
capa_bottom_layer = 5
jj_top_layer = 6
capa_top_layer = 7
pad_top_layer = 8
connection_wire_top_layer = 9
small_jj_top_layer = 10

And the doses we are going to use to write each layer

In [7]:
dose_matric = {str(ground_layer):3,
               str(pad_bottom_layer):15,
               str(connection_wire_bottom_layer):14,
               str(jj_bottom_layer):14,
               str(capa_bottom_layer):12,
               str(jj_top_layer):14,
               str(capa_top_layer):12,
               str(pad_top_layer):15,
               str(connection_wire_top_layer):14,
               str(small_jj_top_layer):18,
              }

Now we can prepare the lithography plan and associate each layer to a job

In [8]:
litho_plan = {'ground':[ground_layer],
              'pad_bottom_layer':[pad_bottom_layer,connection_wire_bottom_layer],
              'bottom_layer':[jj_bottom_layer,capa_bottom_layer],
              'junctions_top_layer':[jj_top_layer, small_jj_top_layer],
              'capacitors_top_layer':[capa_top_layer],
              'pad_top_layer':[pad_top_layer,connection_wire_top_layer],
             }

And now we can define the batch plan, we have to associate to each batch the corresponding jobs and decide the currents used to write each job.
It's also possible to specify the datum and the sleep time in minutes between the jobs

In [9]:
batch_plan = {'ground':{'jobs':['ground'],                                                                                                               
                        'current':[15],
                        'datum':[8],
                        'sleep':[10]},
              'bottom_layer':{'jobs':['bottom_layer','pad_bottom_layer'],
                              'current':[5,15],
                              'datum':[8,8],
                              'sleep':[10,20]},
              'junctions_top_layer':{'jobs':['junctions_top_layer'],
                              'current':[5,15],
                              'datum':[8,8],
                              'sleep':[10,20]},
              'capa_top_layer':{'jobs':['pad_top_layer','capa_top_layer'],
                          'current':[5,15],
                          'datum':[8,8],
                          'sleep':[10,20]},
             }

# Design generation

Now we can create the design, we can both add different devices chip by chip or use for loop to design a all wafer with the same devices.
Let's go chip by chip:
1) We will add a LH TWPA on chip 00
2) Then a RH TWPA on chip 01 and 02
3) And LERs on chip 03

## Composite Right-Left-Handed Josephson Transmission Line (V04)

In [6]:
importlib.reload(TWPA_parameters_cls)
CAD_manager.load_reset_libs()

In [7]:
# Type of device
CAD_manager.device_type = 'CRLH_V04'
device_version = '01'

In [8]:
# Chip
row = 0
column = 0

In [9]:
### Fab parameters file
dirname = os.path.abspath('')
fab_param_filename = os.path.join(dirname, 'default_parameters\\fab_parameters.json')
with open(fab_param_filename) as json_file:
    fab_parameters = json.load(json_file)

In [10]:
### Device parameters file
dirname = os.path.abspath('')
device_param_filename = os.path.join(dirname, 'default_parameters\\device_parameters.json')
with open(device_param_filename) as json_file:
    device_parameter = json.load(json_file)

In [15]:
# CAD manager general info
CAD_manager.modulation = False
CAD_manager.litho_plan = litho_plan
CAD_manager.dose_matric = dose_matric
CAD_manager.chip = str(row)+str(column)

NameError: name 'litho_plan' is not defined

In [ ]:
# Device parameters
params = TWPA_parameters_cls.CRLH_V04_parameters(fab_parameters, device_parameter[CAD_manager.device_type])
print('\n'+'\033[1m'+'Chip '+CAD_manager.chip+'\033[0m')
params.get_params(print_params=True)


Chip 00
Junction capacitance per unit area: c_j = 45.00 fF/um^2
Junction critical current density: j_c = 90.00 A/cm^2
Capacitor capacitance per unit area: c_c = 1.33 fF/um^2
Number of cells: Ncell = 142
Number of junctions in line per cell: Njj_line = 10
Number of junctions to ground per cell: Njj_ground = 10
Number of capacitors in line per cell: Ncapa_line = 2
Number of capacitors to ground per cell: Ncapa_ground = 2
Junction in line dimensions: H = 5.23 um | W = 1.00 um | A = 5.23 um^2
Room temp resistance in line: Rj = 53.91 Ohm
Critical current in line: Ic = 3.62 uA
Normal state resistance in line: Rn = 91.10 Ohm
Josephson inductance in line: Lj_line = 90.89 pH
Josephson capacitance in line: Cj_line = 235.35 fF
Josephson inductance in line per unit cell: Lj_cell_line = 908.94 pH
Josephson capacitance in line per unit cell: Cj_cell_line = 23.53 fF
Junction to ground dimensions: H = 5.23 um | W = 1.00 um | A = 5.23 um^2
Room temp resistance to ground: Rj = 53.91 Ohm
Critical curren

(5.2299999999999995,
 1.0,
 5.2299999999999995,
 1.0,
 20.0,
 5.379999999999999,
 20.0,
 5.379999999999999)

In [ ]:
# Element design definition
CRLH_V03 = TWPA_elements_cls.CRLH_V03_cell()
CRLH_V03.litho_overlap = 0.1
CRLH_V03.etching_offset = 0.25
CRLH_V03.electrode_height_difference_JJ_line = 0.5
CRLH_V03.electrode_height_difference_capa_line = 0.5
CRLH_V03.electrode_height_difference_JJ_ground = 0.5
CRLH_V03.electrode_height_difference_capa_ground = 0.5
CRLH_V03.y_low_current_ground = 3

CRLH_V03.width_jj_line = device_parameter[CAD_manager.device_type]['junction_parameters']['width_of_junction_in_line'] 
CRLH_V03.height_jj_line = device_parameter[CAD_manager.device_type]['junction_parameters']['height_of_junction_in_line']
CRLH_V03.spacing_jj_line = device_parameter[CAD_manager.device_type]['junction_parameters']['spacing_between_junctions_in_line']
CRLH_V03.bottom_spacing_jj_line = device_parameter[CAD_manager.device_type]['junction_parameters']['bottom_spacing_between_junctions_in_line']
CRLH_V03.top_spacing_jj_line = device_parameter[CAD_manager.device_type]['junction_parameters']['top_spacing_between_junctions_in_line']
CRLH_V03.number_of_junctions_line = device_parameter[CAD_manager.device_type]['junction_parameters']['number_of_junctions_per_unit_cell_in_line']

CRLH_V03.width_capa_line = device_parameter[CAD_manager.device_type]['capacitor_parameters']['width_of_capacitor_in_line']
CRLH_V03.height_capa_line = device_parameter[CAD_manager.device_type]['capacitor_parameters']['height_of_capacitor_in_line']
CRLH_V03.spacing_capa_line = device_parameter[CAD_manager.device_type]['capacitor_parameters']['spacing_between_capacitors_in_line']
CRLH_V03.bottom_spacing_capa_line = device_parameter[CAD_manager.device_type]['capacitor_parameters']['bottom_spacing_between_capacitors_in_line']    
CRLH_V03.top_spacing_capa_line = device_parameter[CAD_manager.device_type]['capacitor_parameters']['top_spacing_between_capacitors_in_line']    
CRLH_V03.number_of_capacitors_line = device_parameter[CAD_manager.device_type]['capacitor_parameters']['number_of_capacitors_per_unit_cell_in_line']

CRLH_V03.width_jj_ground = device_parameter[CAD_manager.device_type]['junction_parameters']['width_of_junction_to_ground'] 
CRLH_V03.height_jj_ground = device_parameter[CAD_manager.device_type]['junction_parameters']['height_of_junction_to_ground']
CRLH_V03.spacing_jj_ground = device_parameter[CAD_manager.device_type]['junction_parameters']['spacing_between_junctions_to_ground']
CRLH_V03.bottom_spacing_jj_ground = device_parameter[CAD_manager.device_type]['junction_parameters']['bottom_spacing_between_junctions_to_ground']
CRLH_V03.top_spacing_jj_ground = device_parameter[CAD_manager.device_type]['junction_parameters']['top_spacing_between_junctions_to_ground']
CRLH_V03.number_of_junctions_ground = device_parameter[CAD_manager.device_type]['junction_parameters']['number_of_junctions_per_unit_cell_to_ground']

CRLH_V03.width_capa_ground = device_parameter[CAD_manager.device_type]['capacitor_parameters']['width_of_capacitor_to_ground']
CRLH_V03.height_capa_ground = device_parameter[CAD_manager.device_type]['capacitor_parameters']['height_of_capacitor_to_ground']
CRLH_V03.spacing_capa_ground = device_parameter[CAD_manager.device_type]['capacitor_parameters']['spacing_between_capacitors_to_ground']
CRLH_V03.bottom_spacing_capa_ground = device_parameter[CAD_manager.device_type]['capacitor_parameters']['bottom_spacing_between_capacitors_to_ground']
CRLH_V03.top_spacing_capa_ground = device_parameter[CAD_manager.device_type]['capacitor_parameters']['top_spacing_between_capacitors_to_ground']
CRLH_V03.number_of_capacitors_ground = device_parameter[CAD_manager.device_type]['capacitor_parameters']['number_of_capacitors_per_unit_cell_to_ground']

In [ ]:
#  Layer definition
CRLH_V03.ground_layer = ground_layer
CAD_manager.pad_bottom_layer = pad_bottom_layer
CAD_manager.connection_wire_bottom_layer = connection_wire_bottom_layer
CRLH_V03.jj_bottom_layer = jj_bottom_layer
CRLH_V03.capa_bottom_layer = capa_bottom_layer
CRLH_V03.jj_top_layer = jj_top_layer
CRLH_V03.capa_top_layer = capa_top_layer
CAD_manager.pad_top_layer = pad_top_layer
CAD_manager.connection_wire_top_layer = connection_wire_top_layer
CAD_manager.ground_layer = CRLH_V03.ground_layer

NameError: name 'ground_layer' is not defined

In [ ]:
# CAD manager device info
CAD_manager.n_unit_cells = device_parameter[CAD_manager.device_type]['TWPA_parameters']['number_of_cells']
CAD_manager.element = CRLH_V03
CAD_manager.dev_label = CAD_manager.device_type + '_V' + device_version + '_' + CAD_manager.chip
CAD_manager.design_filename = CAD_manager.device_type + '_' + CAD_manager.chip

KeyError: 'CRLH_V04'

In [20]:
# gds and job file generation
CAD_manager.generate_GDS()
CAD_manager.populate_dose_matric()
CAD_manager.generate_jobs(row=row, column=column)

## Composite Right-Left-Handed Josephson Transmission Line (V02)

In [10]:
importlib.reload(TWPA_parameters_cls)
CAD_manager.load_reset_libs()

In [11]:
# Type of device
CAD_manager.device_type = 'CRLH_V02'
device_version = '01'

In [12]:
# Chip
row = 0
column = 0

In [13]:
### Fab parameters file
dirname = os.path.abspath('')
fab_param_filename = os.path.join(dirname, 'default_parameters\\fab_parameters.json')
with open(fab_param_filename) as json_file:
    fab_parameters = json.load(json_file)

In [14]:
### Device parameters file
dirname = os.path.abspath('')
device_param_filename = os.path.join(dirname, 'default_parameters\\device_parameters.json')
with open(device_param_filename) as json_file:
    device_parameter = json.load(json_file)

In [15]:
# CAD manager general info
CAD_manager.modulation = False
CAD_manager.litho_plan = litho_plan
CAD_manager.dose_matric = dose_matric
CAD_manager.chip = str(row)+str(column)

In [16]:
# Device parameters
params = TWPA_parameters_cls.CRLH_V02_parameters(fab_parameters, device_parameter[CAD_manager.device_type])
print('\n'+'\033[1m'+'Chip '+CAD_manager.chip+'\033[0m')
params.get_params(print_params=True)


Chip 00
Junction capacitance per unit area: c_j = 45.00 fF/um^2
Junction critical current density: j_c = 90.00 A/cm^2
Capacitor capacitance per unit area: c_c = 1.33 fF/um^2
Number of cells: Ncell = 142
Number of junctions in line per cell: Njj_line = 10
Number of junctions to ground per cell: Njj_ground = 10
Number of capacitors in line per cell: Ncapa_line = 2
Number of capacitors to ground per cell: Ncapa_ground = 2
Junction in line dimensions: H = 5.23 um | W = 1.00 um | A = 5.23 um^2
Room temp resistance in line: Rj = 53.91 Ohm
Critical current in line: Ic = 3.62 uA
Normal state resistance in line: Rn = 91.10 Ohm
Josephson inductance in line: Lj_line = 90.89 pH
Josephson capacitance in line: Cj_line = 235.35 fF
Josephson inductance in line per unit cell: Lj_cell_line = 908.94 pH
Josephson capacitance in line per unit cell: Cj_cell_line = 23.53 fF
Junction to ground dimensions: H = 5.23 um | W = 1.00 um | A = 5.23 um^2
Room temp resistance to ground: Rj = 53.91 Ohm
Critical curren

(5.2299999999999995,
 1.0,
 5.2299999999999995,
 1.0,
 20.0,
 5.379999999999999,
 20.0,
 5.379999999999999)

In [17]:
# Element design definition
CRLH_V02 = TWPA_elements_cls.CRLH_V02_cell()
CRLH_V02.litho_overlap = 0.1
CRLH_V02.etching_offset = 0.25
CRLH_V02.electrode_height_difference_JJ_line = 0.5
CRLH_V02.electrode_height_difference_capa_line = 0.5
CRLH_V02.electrode_height_difference_JJ_ground = 0.5
CRLH_V02.electrode_height_difference_capa_ground = 0.5
CRLH_V02.y_low_current_ground = 3

CRLH_V02.width_jj_line = device_parameter[CAD_manager.device_type]['junction_parameters']['width_of_junction_in_line'] 
CRLH_V02.height_jj_line = device_parameter[CAD_manager.device_type]['junction_parameters']['height_of_junction_in_line']
CRLH_V02.spacing_jj_line = device_parameter[CAD_manager.device_type]['junction_parameters']['spacing_between_junctions_in_line']
CRLH_V02.bottom_spacing_jj_line = device_parameter[CAD_manager.device_type]['junction_parameters']['bottom_spacing_between_junctions_in_line']
CRLH_V02.top_spacing_jj_line = device_parameter[CAD_manager.device_type]['junction_parameters']['top_spacing_between_junctions_in_line']
CRLH_V02.number_of_junctions_line = device_parameter[CAD_manager.device_type]['junction_parameters']['number_of_junctions_per_unit_cell_in_line']

CRLH_V02.width_capa_line = device_parameter[CAD_manager.device_type]['capacitor_parameters']['width_of_capacitor_in_line']
CRLH_V02.height_capa_line = device_parameter[CAD_manager.device_type]['capacitor_parameters']['height_of_capacitor_in_line']
CRLH_V02.spacing_capa_line = device_parameter[CAD_manager.device_type]['capacitor_parameters']['spacing_between_capacitors_in_line']
CRLH_V02.bottom_spacing_capa_line = device_parameter[CAD_manager.device_type]['capacitor_parameters']['bottom_spacing_between_capacitors_in_line']    
CRLH_V02.top_spacing_capa_line = device_parameter[CAD_manager.device_type]['capacitor_parameters']['top_spacing_between_capacitors_in_line']    
CRLH_V02.number_of_capacitors_line = device_parameter[CAD_manager.device_type]['capacitor_parameters']['number_of_capacitors_per_unit_cell_in_line']

CRLH_V02.width_jj_ground = device_parameter[CAD_manager.device_type]['junction_parameters']['width_of_junction_to_ground'] 
CRLH_V02.height_jj_ground = device_parameter[CAD_manager.device_type]['junction_parameters']['height_of_junction_to_ground']
CRLH_V02.spacing_jj_ground = device_parameter[CAD_manager.device_type]['junction_parameters']['spacing_between_junctions_to_ground']
CRLH_V02.bottom_spacing_jj_ground = device_parameter[CAD_manager.device_type]['junction_parameters']['bottom_spacing_between_junctions_to_ground']
CRLH_V02.top_spacing_jj_ground = device_parameter[CAD_manager.device_type]['junction_parameters']['top_spacing_between_junctions_to_ground']
CRLH_V02.number_of_junctions_ground = device_parameter[CAD_manager.device_type]['junction_parameters']['number_of_junctions_per_unit_cell_to_ground']

CRLH_V02.width_capa_ground = device_parameter[CAD_manager.device_type]['capacitor_parameters']['width_of_capacitor_to_ground']
CRLH_V02.height_capa_ground = device_parameter[CAD_manager.device_type]['capacitor_parameters']['height_of_capacitor_to_ground']
CRLH_V02.spacing_capa_ground = device_parameter[CAD_manager.device_type]['capacitor_parameters']['spacing_between_capacitors_to_ground']
CRLH_V02.bottom_spacing_capa_ground = device_parameter[CAD_manager.device_type]['capacitor_parameters']['bottom_spacing_between_capacitors_to_ground']
CRLH_V02.top_spacing_capa_ground = device_parameter[CAD_manager.device_type]['capacitor_parameters']['top_spacing_between_capacitors_to_ground']
CRLH_V02.number_of_capacitors_ground = device_parameter[CAD_manager.device_type]['capacitor_parameters']['number_of_capacitors_per_unit_cell_to_ground']

In [18]:
#  Layer definition
CRLH_V02.ground_layer = ground_layer
CAD_manager.pad_bottom_layer = pad_bottom_layer
CAD_manager.connection_wire_bottom_layer = connection_wire_bottom_layer
CRLH_V02.jj_bottom_layer = jj_bottom_layer
CRLH_V02.capa_bottom_layer = capa_bottom_layer
CRLH_V02.jj_top_layer = jj_top_layer
CRLH_V02.capa_top_layer = capa_top_layer
CAD_manager.pad_top_layer = pad_top_layer
CAD_manager.connection_wire_top_layer = connection_wire_top_layer
CAD_manager.ground_layer = CRLH_V02.ground_layer

In [19]:
# CAD manager device info
CAD_manager.n_unit_cells = device_parameter[CAD_manager.device_type]['TWPA_parameters']['number_of_cells']
CAD_manager.element = CRLH_V02
CAD_manager.dev_label = CAD_manager.device_type + '_V' + device_version + '_' + CAD_manager.chip
CAD_manager.design_filename = CAD_manager.device_type + '_' + CAD_manager.chip

In [20]:
# gds and job file generation
CAD_manager.generate_GDS()
CAD_manager.populate_dose_matric()
CAD_manager.generate_jobs(row=row, column=column)

## Composite Right-Left-Handed Josephson Transmission Line (V01)

In [10]:
importlib.reload(TWPA_parameters_cls)
CAD_manager.load_reset_libs()

In [11]:
# Type of device
CAD_manager.device_type = 'CRLH_V01'
device_version = '01'

In [12]:
# Chip
row = 0
column = 0

In [13]:
### Fab parameters file
dirname = os.path.abspath('')
fab_param_filename = os.path.join(dirname, 'default_parameters\\fab_parameters.json')
with open(fab_param_filename) as json_file:
    fab_parameters = json.load(json_file)

In [14]:
### Device parameters file
dirname = os.path.abspath('')
device_param_filename = os.path.join(dirname, 'default_parameters\\device_parameters.json')
with open(device_param_filename) as json_file:
    device_parameter = json.load(json_file)

In [15]:
# CAD manager general info
CAD_manager.modulation = False
CAD_manager.litho_plan = litho_plan
CAD_manager.dose_matric = dose_matric
CAD_manager.chip = str(row)+str(column)

In [16]:
# Device parameters
params = TWPA_parameters_cls.CRLH_V01_parameters(fab_parameters, device_parameter[CAD_manager.device_type])
print('\n'+'\033[1m'+'Chip '+CAD_manager.chip+'\033[0m')
params.get_params(print_params=True)


Chip 00
Junction capacitance per unit area: c_j = 45.00 fF/um^2
Junction critical current density: j_c = 90.00 A/cm^2
Capacitor capacitance per unit area: c_c = 1.33 fF/um^2
Number of cells: Ncell = 128
Number of junctions in line per cell: Njj_line = 10
Number of junctions to ground per cell: Njj_ground = 10
Number of capacitors in line per cell: Ncapa_line = 2
Number of capacitors to ground per cell: Ncapa_ground = 2
Junction in line dimensions: H = 5.23 um | W = 1.00 um | A = 5.23 um^2
Room temp resistance in line: Rj = 53.91 Ohm
Critical current in line: Ic = 3.62 uA
Normal state resistance in line: Rn = 91.10 Ohm
Josephson inductance in line: Lj_line = 90.89 pH
Josephson capacitance in line: Cj_line = 235.35 fF
Josephson inductance in line per unit cell: Lj_cell_line = 908.94 pH
Josephson capacitance in line per unit cell: Cj_cell_line = 23.53 fF
Junction to ground dimensions: H = 5.23 um | W = 1.00 um | A = 5.23 um^2
Room temp resistance to ground: Rj = 53.91 Ohm
Critical curren

(5.2299999999999995,
 1.0,
 5.2299999999999995,
 1.0,
 20.0,
 5.379999999999999,
 20.0,
 5.379999999999999)

In [ ]:
# Element design definition
CRLH_V01 = TWPA_elements_cls.CRLH_V01_cell()
CRLH_V01.litho_overlap = 0.1
CRLH_V01.etching_offset = 0.25
CRLH_V01.electrode_height_difference_JJ_line = 0.5
CRLH_V01.electrode_height_difference_capa_line = 0.5
CRLH_V01.electrode_height_difference_JJ_ground = 0.5
CRLH_V01.electrode_height_difference_capa_ground = 0.5
CRLH_V01.y_low_current_ground = 3

CRLH_V01.width_jj_line = device_parameter[CAD_manager.device_type]['junction_parameters']['width_of_junction_in_line'] 
CRLH_V01.height_jj_line = device_parameter[CAD_manager.device_type]['junction_parameters']['height_of_junction_in_line']
CRLH_V01.spacing_jj_line = device_parameter[CAD_manager.device_type]['junction_parameters']['spacing_between_junctions_in_line']
CRLH_V01.bottom_spacing_jj_line = device_parameter[CAD_manager.device_type]['junction_parameters']['bottom_spacing_between_junctions_in_line']
CRLH_V01.top_spacing_jj_line = device_parameter[CAD_manager.device_type]['junction_parameters']['top_spacing_between_junctions_in_line']
CRLH_V01.number_of_junctions_line = device_parameter[CAD_manager.device_type]['junction_parameters']['number_of_junctions_per_unit_cell_in_line']

CRLH_V01.width_capa_line = device_parameter[CAD_manager.device_type]['capacitor_parameters']['width_of_capacitor_in_line']
CRLH_V01.height_capa_line = device_parameter[CAD_manager.device_type]['capacitor_parameters']['height_of_capacitor_in_line']
CRLH_V01.spacing_capa_line = device_parameter[CAD_manager.device_type]['capacitor_parameters']['spacing_between_capacitors_in_line']
CRLH_V01.bottom_spacing_capa_line = device_parameter[CAD_manager.device_type]['capacitor_parameters']['bottom_spacing_between_capacitors_in_line']    
CRLH_V01.top_spacing_capa_line = device_parameter[CAD_manager.device_type]['capacitor_parameters']['top_spacing_between_capacitors_in_line']    
CRLH_V01.number_of_capacitors_line = device_parameter[CAD_manager.device_type]['capacitor_parameters']['number_of_capacitors_per_unit_cell_in_line']

CRLH_V01.width_jj_ground = device_parameter[CAD_manager.device_type]['junction_parameters']['width_of_junction_to_ground'] 
CRLH_V01.height_jj_ground = device_parameter[CAD_manager.device_type]['junction_parameters']['height_of_junction_to_ground']
CRLH_V01.spacing_jj_ground = device_parameter[CAD_manager.device_type]['junction_parameters']['spacing_between_junctions_to_ground']
CRLH_V01.bottom_spacing_jj_ground = device_parameter[CAD_manager.device_type]['junction_parameters']['bottom_spacing_between_junctions_to_ground']
CRLH_V01.top_spacing_jj_ground = device_parameter[CAD_manager.device_type]['junction_parameters']['top_spacing_between_junctions_to_ground']
CRLH_V01.number_of_junctions_ground = device_parameter[CAD_manager.device_type]['junction_parameters']['number_of_junctions_per_unit_cell_to_ground']

CRLH_V01.width_capa_ground = device_parameter[CAD_manager.device_type]['capacitor_parameters']['width_of_capacitor_to_ground']
CRLH_V01.height_capa_ground = device_parameter[CAD_manager.device_type]['capacitor_parameters']['height_of_capacitor_to_ground']
CRLH_V01.spacing_capa_ground = device_parameter[CAD_manager.device_type]['capacitor_parameters']['spacing_between_capacitors_to_ground']
CRLH_V01.bottom_spacing_capa_ground = device_parameter[CAD_manager.device_type]['capacitor_parameters']['bottom_spacing_between_capacitors_to_ground']
CRLH_V01.top_spacing_capa_ground = device_parameter[CAD_manager.device_type]['capacitor_parameters']['top_spacing_between_capacitors_to_ground']
CRLH_V01.number_of_capacitors_ground = device_parameter[CAD_manager.device_type]['capacitor_parameters']['number_of_capacitors_per_unit_cell_to_ground']

In [18]:
#  Layer definition
CRLH_V01.ground_layer = ground_layer
CAD_manager.pad_bottom_layer = pad_bottom_layer
CAD_manager.connection_wire_bottom_layer = connection_wire_bottom_layer
CRLH_V01.jj_bottom_layer = jj_bottom_layer
CRLH_V01.capa_bottom_layer = capa_bottom_layer
CRLH_V01.jj_top_layer = jj_top_layer
CRLH_V01.capa_top_layer = capa_top_layer
CAD_manager.pad_top_layer = pad_top_layer
CAD_manager.connection_wire_top_layer = connection_wire_top_layer
CAD_manager.ground_layer = CRLH_V01.ground_layer

In [19]:
# CAD manager device info
CAD_manager.n_unit_cells = device_parameter[CAD_manager.device_type]['TWPA_parameters']['number_of_cells']
CAD_manager.element = CRLH_V01
CAD_manager.dev_label = CAD_manager.device_type + '_V' + device_version + '_' + CAD_manager.chip
CAD_manager.design_filename = CAD_manager.device_type + '_' + CAD_manager.chip

In [20]:
# gds and job file generation
CAD_manager.generate_GDS()
CAD_manager.populate_dose_matric()
CAD_manager.generate_jobs(row=row, column=column)

## Left-Handed Josephson Transmission Line

In [21]:
importlib.reload(TWPA_parameters_cls)
CAD_manager.load_reset_libs()

In [22]:
# Type of device
CAD_manager.device_type = 'LH'
device_version = '01'

In [23]:
# Chip
row = 0
column = 0

In [24]:
### Fab parameters file
dirname = os.path.abspath('')
fab_param_filename = os.path.join(dirname, 'default_parameters\\fab_parameters.json')
with open(fab_param_filename) as json_file:
    fab_parameters = json.load(json_file)


In [25]:
### Device parameters file
dirname = os.path.abspath('')
device_param_filename = os.path.join(dirname, 'default_parameters\\device_parameters.json')
with open(device_param_filename) as json_file:
    device_parameter = json.load(json_file)

In [26]:
# CAD manager general info
CAD_manager.modulation = False
CAD_manager.litho_plan = litho_plan
CAD_manager.dose_matric = dose_matric
CAD_manager.chip = str(row)+str(column)

In [27]:
# Device parameters
params = TWPA_parameters_cls.LH_parameters(fab_parameters, device_parameter[CAD_manager.device_type])
print('\n'+'\033[1m'+'Chip '+CAD_manager.chip+'\033[0m')
number_of_junctions, area_capa, f0 = params.get_params(print_params=True)


Chip 00
 Parameters 
Junction capacitance per unit area: c_j = 45.00 fF/um^2
Junction critical current density: j_c = 90.00 A/cm^2
Capacitor capacitance per unit area: c_c = 7.44 fF/um^2
Number of cells: Ncell = 600
Number of junctions to ground: Nj_ground = 26
Number of capacitors per cell: Ncapa = 2
Junction dimensions: H = 7.00 um | W = 1.00 um | A = 7.00 um^2
Room temp resistance: Rj = 40.28 Ohm
Critical current: Ic = 4.85 uA
Normal state resistance: Rn = 68.07 Ohm
Josephson inductance: Lj = 67.91 pH
Josephson capacitance: Cj = 315.00 fF
Inductance to ground: Lj_ground = 1.77 nH
Capacitance to ground: Cj_ground = 12.12 fF
Series capacitance: C = 706.27 fF
Capacitor area: A = 189.86 um^2
Plasma frequency: fj = 34.41 GHz
Cut-off frequency: f0 = 4.51 GHz


In [17]:
# Element design definition
LH = TWPA_elements_cls.LHcell()
LH.litho_overlap = 0.1
LH.etching_offset = 0.5
LH.electrode_height_difference_capa = 5
LH.electrode_height_difference_JJ = 0.5
LH.y_low_current_ground = 3
LH.number_of_capacitors = int(device_parameter[CAD_manager.device_type]['capacitor_parameters']['number_of_capacitors_per_unit_cell'])
LH.width_jj = device_parameter[CAD_manager.device_type]['junction_parameters']['width_of_junction'] 
LH.height_jj = device_parameter[CAD_manager.device_type]['junction_parameters']['height_of_junction']
LH.spacing_jj = device_parameter[CAD_manager.device_type]['junction_parameters']['spacing_between_junctions']
LH.number_of_junctions = number_of_junctions
LH.width_capa = device_parameter[CAD_manager.device_type]['capacitor_parameters']['width_of_capacitor']
LH.height_capa = area_capa/LH.width_capa
LH.new_area_capa = area_capa
LH.spacing_capa = device_parameter[CAD_manager.device_type]['capacitor_parameters']['spacing_between_capa']    

In [18]:
#  Layer definition
LH.ground_layer = ground_layer
CAD_manager.pad_bottom_layer = pad_bottom_layer
CAD_manager.connection_wire_bottom_layer = connection_wire_bottom_layer
LH.jj_bottom_layer = jj_bottom_layer
LH.capa_bottom_layer = capa_bottom_layer
LH.jj_top_layer = jj_top_layer
LH.capa_top_layer = capa_top_layer
CAD_manager.pad_top_layer = pad_top_layer
CAD_manager.connection_wire_top_layer = connection_wire_top_layer
CAD_manager.ground_layer = LH.ground_layer

In [19]:
# CAD manager device info
CAD_manager.n_unit_cells = device_parameter[CAD_manager.device_type]['TWPA_parameters']['number_of_cells']
CAD_manager.element = LH
CAD_manager.dev_label = CAD_manager.device_type + '_V' + device_version + '_' + CAD_manager.chip
CAD_manager.dev_label += ' | ' + str(CAD_manager.n_unit_cells) + ' | ' + str(f0)
CAD_manager.design_filename = CAD_manager.device_type + '_' + CAD_manager.chip

In [20]:
# gds and job file generation
CAD_manager.generate_GDS()
CAD_manager.populate_dose_matric()
CAD_manager.generate_jobs(row=row, column=column)

## Right-Handed Josephson Transmission Line

### With parallel plate capacitors

#### No modulation

In [21]:
importlib.reload(TWPA_parameters_cls)
CAD_manager.load_reset_libs()

In [22]:
# Type of device
CAD_manager.device_type = 'RH'
device_version = '01'

In [23]:
# Chip
row = 0
column = 1

In [24]:
### Fab parameters file
dirname = os.path.abspath('')
fab_param_filename = os.path.join(dirname, 'default_parameters\\fab_parameters.json')
with open(fab_param_filename) as json_file:
    fab_parameters = json.load(json_file)

In [25]:
### Device parameters file
dirname = os.path.abspath('')
device_param_filename = os.path.join(dirname, 'default_parameters\\device_parameters.json')
with open(device_param_filename) as json_file:
    device_parameter = json.load(json_file)

In [26]:
# CAD manager general info
CAD_manager.modulation = False
CAD_manager.litho_plan = litho_plan
CAD_manager.dose_matric = dose_matric
CAD_manager.chip = str(row)+str(column)

In [27]:
# Device parameters
params = TWPA_parameters_cls.RH_parameters(fab_parameters, device_parameter[CAD_manager.device_type])
print('\n'+'\033[1m'+'Chip '+CAD_manager.chip+'\033[0m')
area_capa = params.get_params(print_params=True)


Chip 01
Junction capacitance per unit area: c_j = 45.00 fF/um^2
Junction critical current density: j_c = 50.00 A/cm^2
Capacitor capacitance per unit area: c_c = 8.63 fF/um^2
Number of cells: Ncell = 900
Number of junctions per cell: Njj = 2
Number of capacitors per cell: Ncapa = 2
Junction dimensions: H = 3.50 um | W = 2.00 um | A = 7.00 um^2
Room temp resistance: Rj = 72.50 Ohm
Critical current: Ic = 2.69 uA
Normal state resistance: Rn = 122.52 Ohm
Josephson inductance: Lj = 122.24 pH
Josephson capacitance: Cj = 315.00 fF
Josephson inductance per unit cell: Lj_cell = 244.48 pH
Josephson capacitance per unit cell: Cj_cell = 157.50 fF
Ground capacitance to get 50 Ohm matching: Cg = 97.79 fF
Capacitor area: A = 11.33 um^2
Plasma frequency: fj = 25.65 GHz
Cut-off frequency: f0 = 32.55 GHz
No modulation


In [28]:
# Element design definition
RH = TWPA_elements_cls.RHcell()
RH.litho_overlap = 0.1
RH.etching_offset = 0.25
RH.electrode_height_difference_jj = 0.5
RH.y_low_current_ground = 3
RH.width_jj = device_parameter[CAD_manager.device_type]['junction_parameters']['width_of_junction'] 
RH.height_jj = device_parameter[CAD_manager.device_type]['junction_parameters']['height_of_junction']
RH.spacing_jj = device_parameter[CAD_manager.device_type]['junction_parameters']['spacing_between_junctions'] 
RH.number_of_junctions = device_parameter[CAD_manager.device_type]['junction_parameters']['number_of_junctions_per_unit_cell'] 
RH.area_capa = area_capa
RH.spacing_capa = device_parameter[CAD_manager.device_type]['capacitor_parameters']['spacing_between_capacitors'] 
RH.electrode_height_difference_capa = 1
RH.number_of_capacitors = device_parameter[CAD_manager.device_type]['capacitor_parameters']['number_of_capacitors_per_unit_cell'] 


In [29]:
# Layers definition
RH.ground_layer = ground_layer
CAD_manager.pad_bottom_layer = pad_bottom_layer
CAD_manager.connection_wire_bottom_layer = connection_wire_bottom_layer
RH.jj_bottom_layer = jj_bottom_layer
RH.capa_bottom_layer = capa_bottom_layer
RH.jj_top_layer = jj_top_layer
RH.capa_top_layer = capa_top_layer
CAD_manager.pad_top_layer = pad_top_layer
CAD_manager.connection_wire_top_layer = connection_wire_top_layer
CAD_manager.ground_layer = RH.ground_layer

In [30]:
# CAD manager device info
CAD_manager.n_unit_cells = device_parameter[CAD_manager.device_type]['TWPA_parameters']['number_of_cells']
CAD_manager.element = RH
CAD_manager.dev_label = CAD_manager.device_type + '_V' + device_version + '_' + CAD_manager.chip
CAD_manager.dev_label += ' | ' + str(CAD_manager.n_unit_cells)
CAD_manager.design_filename = CAD_manager.device_type + '_' + CAD_manager.chip

In [31]:
# gds and job files generation
CAD_manager.generate_GDS()
CAD_manager.populate_dose_matric()
CAD_manager.generate_jobs(row=row, column=column)

##### Change area capacitors

Let's imagine we already deposited our junctions and did the DC test from which we extracted a different crittical current density respect to the one reported in the fab_parameter json file.
It's possible to change the area of the capacitors without changing anything else in the design to still have 50 Ohm matching. To do so we need to update the value of the jc, pass the previous value of the capacitor area to the variable element.area_capa and set the change_area_capa variable to True.

In [32]:
importlib.reload(TWPA_parameters_cls)
CAD_manager.load_reset_libs()

In [33]:
# New parameters
change_area_capa = True
new_jc = 39 #A/cm^2

# Old parameters
old_area_capa = 10.9

In [34]:
# Type of device
CAD_manager.device_type = 'RH'
device_version = '01'

In [35]:
# Chip
row = 0
column = 1

In [36]:
### Fab parameters file
dirname = os.path.abspath('')
fab_param_filename = os.path.join(dirname, 'default_parameters\\fab_parameters.json')
with open(fab_param_filename) as json_file:
    fab_parameters = json.load(json_file)

In [37]:
### Device parameters file
dirname = os.path.abspath('')
device_param_filename = os.path.join(dirname, 'default_parameters\\device_parameters.json')
with open(device_param_filename) as json_file:
    device_parameter = json.load(json_file)

In [38]:
# Update parameter
fab_parameters['junction_parameters']['critical_current_density'] = new_jc

In [39]:
# CAD manager general info
CAD_manager.modulation = False
CAD_manager.litho_plan = litho_plan
CAD_manager.dose_matric = dose_matric
CAD_manager.chip = str(row)+str(column)

In [40]:
# Device parameters
params = TWPA_parameters_cls.RH_parameters(fab_parameters, device_parameter[CAD_manager.device_type])
print('\n'+'\033[1m'+'Chip '+CAD_manager.chip+'\033[0m')
area_capa = params.get_params(print_params=True)


Chip 01
Junction capacitance per unit area: c_j = 45.00 fF/um^2
Junction critical current density: j_c = 39.00 A/cm^2
Capacitor capacitance per unit area: c_c = 8.63 fF/um^2
Number of cells: Ncell = 900
Number of junctions per cell: Njj = 2
Number of capacitors per cell: Ncapa = 2
Junction dimensions: H = 3.50 um | W = 2.00 um | A = 7.00 um^2
Room temp resistance: Rj = 92.95 Ohm
Critical current: Ic = 2.10 uA
Normal state resistance: Rn = 157.08 Ohm
Josephson inductance: Lj = 156.72 pH
Josephson capacitance: Cj = 315.00 fF
Josephson inductance per unit cell: Lj_cell = 313.43 pH
Josephson capacitance per unit cell: Cj_cell = 157.50 fF
Ground capacitance to get 50 Ohm matching: Cg = 125.37 fF
Capacitor area: A = 14.53 um^2
Plasma frequency: fj = 22.65 GHz
Cut-off frequency: f0 = 25.39 GHz
No modulation


In [41]:
# Element design definition
RH = TWPA_elements_cls.RHcell()

RH.litho_overlap = 0.1
RH.etching_offset = 0.25
RH.electrode_height_difference_jj = 0.5
RH.y_low_current_ground = 3
RH.width_jj = device_parameter[CAD_manager.device_type]['junction_parameters']['width_of_junction'] 
RH.height_jj = device_parameter[CAD_manager.device_type]['junction_parameters']['height_of_junction']
RH.spacing_jj = device_parameter[CAD_manager.device_type]['junction_parameters']['spacing_between_junctions'] 
RH.number_of_junctions = device_parameter[CAD_manager.device_type]['junction_parameters']['number_of_junctions_per_unit_cell'] 
RH.spacing_capa = device_parameter[CAD_manager.device_type]['capacitor_parameters']['spacing_between_capacitors'] 
RH.electrode_height_difference_capa = 1
RH.number_of_capacitors = device_parameter[CAD_manager.device_type]['capacitor_parameters']['number_of_capacitors_per_unit_cell'] 
RH.area_capa = old_area_capa
RH.new_area_capa = area_capa

In [42]:
# Layer definition
RH.ground_layer = ground_layer
CAD_manager.pad_bottom_layer = pad_bottom_layer
CAD_manager.connection_wire_bottom_layer = connection_wire_bottom_layer
RH.jj_bottom_layer = jj_bottom_layer
RH.capa_bottom_layer = capa_bottom_layer
RH.jj_top_layer = jj_top_layer
RH.capa_top_layer = capa_top_layer
CAD_manager.pad_top_layer = pad_top_layer
CAD_manager.connection_wire_top_layer = connection_wire_top_layer
CAD_manager.ground_layer = RH.ground_layer

In [43]:
# CAD manager device info
CAD_manager.n_unit_cells = device_parameter[CAD_manager.device_type]['TWPA_parameters']['number_of_cells']
CAD_manager.element = RH
CAD_manager.dev_label = CAD_manager.device_type + '_V' + device_version + '_' + CAD_manager.chip
CAD_manager.dev_label += ' | ' + str(CAD_manager.n_unit_cells)
CAD_manager.design_filename = CAD_manager.device_type + '_' + CAD_manager.chip

In [44]:
# gds and job file generation
CAD_manager.generate_GDS(change_area_capa=change_area_capa)
CAD_manager.populate_dose_matric()
CAD_manager.generate_jobs(row=row, column=column)

#### Impedance modulation

In [45]:
importlib.reload(TWPA_parameters_cls)
CAD_manager.load_reset_libs()

In [46]:
TWPA_modulation = True

In [47]:
# Type of device
CAD_manager.device_type = 'RH'
device_version = '01'

In [48]:
# Chip
row = 0
column = 2

In [49]:
### Fab parameters file
dirname = os.path.abspath('')
fab_param_filename = os.path.join(dirname, 'default_parameters\\fab_parameters.json')
with open(fab_param_filename) as json_file:
    fab_parameters = json.load(json_file)

In [50]:
### Device parameters file
dirname = os.path.abspath('')
device_param_filename = os.path.join(dirname, 'default_parameters\\device_parameters.json')
with open(device_param_filename) as json_file:
    device_parameter = json.load(json_file)

In [51]:
# CAD manager general info
CAD_manager.modulation = TWPA_modulation
CAD_manager.litho_plan = litho_plan
CAD_manager.dose_matric = dose_matric
CAD_manager.chip = str(row)+str(column)

In [52]:
# Device parameters
params = TWPA_parameters_cls.RH_parameters(fab_parameters, device_parameter[CAD_manager.device_type])
print('\n'+'\033[1m'+'Chip '+CAD_manager.chip+'\033[0m')
area_capa = params.get_params(modulation=CAD_manager.modulation, print_params=True)


Chip 02
Junction capacitance per unit area: c_j = 45.00 fF/um^2
Junction critical current density: j_c = 50.00 A/cm^2
Capacitor capacitance per unit area: c_c = 8.63 fF/um^2
Number of cells: Ncell = 900
Number of junctions per cell: Njj = 2
Number of capacitors per cell: Ncapa = 2
Junction dimensions: H = 3.50 um | W = 2.00 um | A = 7.00 um^2
Room temp resistance: Rj = 72.50 Ohm
Critical current: Ic = 2.69 uA
Normal state resistance: Rn = 122.52 Ohm
Josephson inductance: Lj = 122.24 pH
Josephson capacitance: Cj = 315.00 fF
Josephson inductance per unit cell: Lj_cell = 244.48 pH
Josephson capacitance per unit cell: Cj_cell = 157.50 fF
Ground capacitance to get 50 Ohm matching: Cg = 97.79 fF
Capacitor area: A = 11.33 um^2
Plasma frequency: fj = 25.65 GHz
Cut-off frequency: f0 = 32.55 GHz
Modulation period: Np = 16
Modulation amplitude: eta = 5%
Gap frequency: fgap = 6.20 GHz


In [53]:
# Element design definition
RH = TWPA_elements_cls.RHcell()

RH.litho_overlap = 0.1
RH.etching_offset = 0.25
RH.electrode_height_difference_jj = 0.5
RH.y_low_current_ground = 3
RH.width_jj = device_parameter[CAD_manager.device_type]['junction_parameters']['width_of_junction'] 
RH.height_jj = device_parameter[CAD_manager.device_type]['junction_parameters']['height_of_junction']
RH.spacing_jj = device_parameter[CAD_manager.device_type]['junction_parameters']['spacing_between_junctions'] 
RH.number_of_junctions = device_parameter[CAD_manager.device_type]['junction_parameters']['number_of_junctions_per_unit_cell'] 
RH.area_capa = area_capa
RH.spacing_capa = device_parameter[CAD_manager.device_type]['capacitor_parameters']['spacing_between_capacitors'] 
RH.electrode_height_difference_capa = 1
RH.number_of_capacitors = device_parameter[CAD_manager.device_type]['capacitor_parameters']['number_of_capacitors_per_unit_cell'] 

In [54]:
# Layer definition
RH.ground_layer = ground_layer
CAD_manager.pad_bottom_layer = pad_bottom_layer
CAD_manager.connection_wire_bottom_layer = connection_wire_bottom_layer
RH.jj_bottom_layer = jj_bottom_layer
RH.capa_bottom_layer = capa_bottom_layer
RH.jj_top_layer = jj_top_layer
RH.capa_top_layer = capa_top_layer
CAD_manager.pad_top_layer = pad_top_layer
CAD_manager.connection_wire_top_layer = connection_wire_top_layer
CAD_manager.ground_layer = RH.ground_layer

In [55]:
# CAD manager device info
CAD_manager.element = RH
CAD_manager.n_unit_cells = device_parameter[CAD_manager.device_type]['TWPA_parameters']['number_of_cells']
CAD_manager.modulation_period = device_parameter[CAD_manager.device_type]['TWPA_parameters']['modulation_period']
CAD_manager.modulation_amplitude_percent = device_parameter[CAD_manager.device_type]['TWPA_parameters']['modulation_amplitude_percent']
CAD_manager.dev_label = CAD_manager.device_type + '_V' + device_version + '_' + CAD_manager.chip
CAD_manager.dev_label += ' | ' + str(CAD_manager.n_unit_cells) + ' | '+str(CAD_manager.modulation_period)+' | '+str(CAD_manager.modulation_amplitude_percent)+'%'
CAD_manager.design_filename = CAD_manager.device_type + '_' + CAD_manager.chip

In [56]:
# gds and job file generation
CAD_manager.generate_GDS()
CAD_manager.populate_dose_matric()
CAD_manager.generate_jobs(row=row, column=column)

### With top ground

If a given device on the wafer has different number of layers and/or doses we can change the dose_matric variable

In [57]:
dose_matric = {str(ground_layer):3,
               str(pad_bottom_layer):15,
               str(connection_wire_bottom_layer):14,
               str(jj_bottom_layer):14,
               str(jj_top_layer):14,
               str(pad_top_layer):15,
               str(connection_wire_top_layer):14,
              }

#### No modulation

In [58]:
importlib.reload(TWPA_parameters_cls)
CAD_manager.load_reset_libs()

In [59]:
# Type of device
CAD_manager.device_type = 'SJ'
device_version = '01'

In [60]:
# Chip
row = 0
column = 3

In [61]:
### Fab parameters file
dirname = os.path.abspath('')
fab_param_filename = os.path.join(dirname, 'default_parameters\\fab_parameters.json')
with open(fab_param_filename) as json_file:
    fab_parameters = json.load(json_file)

In [62]:
### Device parameters file
dirname = os.path.abspath('')
device_param_filename = os.path.join(dirname, 'default_parameters\\device_parameters.json')
with open(device_param_filename) as json_file:
    device_parameter = json.load(json_file)

In [63]:
# CAD manager general info
CAD_manager.modulation = False
CAD_manager.litho_plan = litho_plan
CAD_manager.dose_matric = dose_matric
CAD_manager.chip = str(row)+str(column)

In [64]:
# Device parameters
params = TWPA_parameters_cls.SJ_parameters(fab_parameters, device_parameter[CAD_manager.device_type])
print('\n'+'\033[1m'+'Chip '+CAD_manager.chip+'\033[0m')
Cg = params.get_params(print_params=True)


Chip 03
Junction capacitance per unit area: c_j = 45.00 fF/um^2
Junction critical current density: j_c = 50.00 A/cm^2
Number of cells: Ncell = 2000
Number of junctions per cell: Njj = 1
Junction dimensions: H = 7.00 um | W = 1.00 um | A = 7.00 um^2
Room temp resistance: Rj = 72.50 Ohm
Critical current: Ic = 2.69 uA
Normal state resistance: Rn = 122.52 Ohm
Josephson inductance: Lj = 122.24 pH
Josephson capacitance: Cj = 315.00 fF
Josephson inductance per unit cell: Lj_cell = 122.24 pH
Josephson capacitance per unit cell: Cj_cell = 315.00 fF
Ground capacitance to get 50 Ohm matching: Cg = 48.90 fF
Plasma frequency: fj = 25.65 GHz
Cut-off frequency: f0 = 65.10 GHz
No modulation


In [65]:
# Element design definition
SJ = TWPA_elements_cls.Overlap_jj()
SJ.litho_overlap = 0.1
SJ.etching_offset = 0.25
SJ.electrode_height_difference = 0.5
SJ.width_jj = device_parameter[CAD_manager.device_type]['junction_parameters']['width_of_junction'] 
SJ.height_jj = device_parameter[CAD_manager.device_type]['junction_parameters']['height_of_junction']
SJ.spacing_jj = device_parameter[CAD_manager.device_type]['junction_parameters']['spacing_between_junctions'] 

In [66]:
# Layers definition
CAD_manager.pad_bottom_layer = pad_bottom_layer
CAD_manager.connection_wire_bottom_layer = connection_wire_bottom_layer
SJ.jj_bottom_layer = jj_bottom_layer
SJ.jj_top_layer = jj_top_layer
CAD_manager.pad_top_layer = pad_top_layer
CAD_manager.connection_wire_top_layer = connection_wire_top_layer
CAD_manager.ground_layer = ground_layer

In [67]:
# CAD manager device info
CAD_manager.n_unit_cells = device_parameter[CAD_manager.device_type]['TWPA_parameters']['number_of_cells']
CAD_manager.element = SJ
CAD_manager.dev_label = CAD_manager.device_type + '_V' + device_version + '_' + CAD_manager.chip
CAD_manager.dev_label += ' | ' + str(CAD_manager.n_unit_cells)
CAD_manager.design_filename = CAD_manager.device_type + '_' + CAD_manager.chip

In [68]:
# gds and job files generation
CAD_manager.generate_GDS()
CAD_manager.populate_dose_matric()
CAD_manager.generate_jobs(row=row, column=column)

#### Impedance modulation

In [69]:
importlib.reload(TWPA_parameters_cls)
CAD_manager.load_reset_libs()

In [70]:
TWPA_modulation = True

In [71]:
# Type of device
CAD_manager.device_type = 'SJ'
device_version = '01'

In [72]:
# Chip
row = 1
column = 0

In [73]:
### Fab parameters file
dirname = os.path.abspath('')
fab_param_filename = os.path.join(dirname, 'default_parameters\\fab_parameters.json')
with open(fab_param_filename) as json_file:
    fab_parameters = json.load(json_file)

In [74]:
### Device parameters file
dirname = os.path.abspath('')
device_param_filename = os.path.join(dirname, 'default_parameters\\device_parameters.json')
with open(device_param_filename) as json_file:
    device_parameter = json.load(json_file)

In [75]:
# CAD manager general info
CAD_manager.modulation = TWPA_modulation
CAD_manager.litho_plan = litho_plan
CAD_manager.dose_matric = dose_matric
CAD_manager.chip = str(row)+str(column)

In [76]:
# Device parameters
params = TWPA_parameters_cls.SJ_parameters(fab_parameters, device_parameter[CAD_manager.device_type])
print('\n'+'\033[1m'+'Chip '+CAD_manager.chip+'\033[0m')
Cg = params.get_params(modulation=CAD_manager.modulation, print_params=True)


Chip 10
Junction capacitance per unit area: c_j = 45.00 fF/um^2
Junction critical current density: j_c = 50.00 A/cm^2
Number of cells: Ncell = 2000
Number of junctions per cell: Njj = 1
Junction dimensions: H = 7.00 um | W = 1.00 um | A = 7.00 um^2
Room temp resistance: Rj = 72.50 Ohm
Critical current: Ic = 2.69 uA
Normal state resistance: Rn = 122.52 Ohm
Josephson inductance: Lj = 122.24 pH
Josephson capacitance: Cj = 315.00 fF
Josephson inductance per unit cell: Lj_cell = 122.24 pH
Josephson capacitance per unit cell: Cj_cell = 315.00 fF
Ground capacitance to get 50 Ohm matching: Cg = 48.90 fF
Plasma frequency: fj = 25.65 GHz
Cut-off frequency: f0 = 65.10 GHz
Modulation period: Np = 16
Modulation amplitude: eta = 5%
Gap frequency: fgap = 11.44 GHz


In [77]:
# Element design definition
SJ = TWPA_elements_cls.Overlap_jj()
SJ.litho_overlap = 0.1
SJ.etching_offset = 0.25
SJ.electrode_height_difference = 0.5
SJ.width_jj = device_parameter[CAD_manager.device_type]['junction_parameters']['width_of_junction'] 
SJ.height_jj = device_parameter[CAD_manager.device_type]['junction_parameters']['height_of_junction']
SJ.spacing_jj = device_parameter[CAD_manager.device_type]['junction_parameters']['spacing_between_junctions'] 

In [78]:
# Layer definition
CAD_manager.pad_bottom_layer = pad_bottom_layer
CAD_manager.connection_wire_bottom_layer = connection_wire_bottom_layer
SJ.jj_bottom_layer = jj_bottom_layer
SJ.jj_top_layer = jj_top_layer
CAD_manager.pad_top_layer = pad_top_layer
CAD_manager.connection_wire_top_layer = connection_wire_top_layer
CAD_manager.ground_layer = ground_layer

In [79]:
# CAD manager device info
CAD_manager.element = SJ
CAD_manager.n_unit_cells = device_parameter[CAD_manager.device_type]['TWPA_parameters']['number_of_cells']
CAD_manager.modulation_period = device_parameter[CAD_manager.device_type]['TWPA_parameters']['modulation_period']
CAD_manager.modulation_amplitude_percent = device_parameter[CAD_manager.device_type]['TWPA_parameters']['modulation_amplitude_percent']
CAD_manager.dev_label = CAD_manager.device_type + '_V' + device_version + '_' + CAD_manager.chip
CAD_manager.dev_label += ' | ' + str(CAD_manager.n_unit_cells) + ' | '+str(CAD_manager.modulation_period)+' | '+str(CAD_manager.modulation_amplitude_percent)+'%'
CAD_manager.design_filename = CAD_manager.device_type + '_' + CAD_manager.chip

In [80]:
# gds and job file generation
CAD_manager.generate_GDS()
CAD_manager.populate_dose_matric()
CAD_manager.generate_jobs(row=row, column=column)

## Resonator

### Lumped element

In [9]:
importlib.reload(TWPA_parameters_cls)
CAD_manager.load_reset_libs()

In [10]:
dose_matric = {str(ground_layer):3,
               str(capa_top_layer):14,
               str(pad_top_layer):15,
               str(connection_wire_top_layer):14,
              }

In [11]:
# Type of device
CAD_manager.device_type = 'LER'
device_version = '01'

In [12]:
# Chip
row = 1
column = 2

In [13]:
### Fab parameters file
dirname = os.path.abspath('')
fab_param_filename = os.path.join(dirname, 'default_parameters\\fab_parameters.json')
with open(fab_param_filename) as json_file:
    fab_parameters = json.load(json_file)

In [14]:
### Device parameters file
dirname = os.path.abspath('')
device_param_filename = os.path.join(dirname, 'default_parameters\\device_parameters.json')
with open(device_param_filename) as json_file:
    device_parameter = json.load(json_file)

In [15]:
# CAD manager general info
CAD_manager.modulation = False
CAD_manager.chip = str(row)+str(column)
CAD_manager.litho_plan = litho_plan
CAD_manager.dose_matric = dose_matric

In [16]:
# Device parameters
params = TWPA_parameters_cls.LER_parameters(fab_parameters, device_parameter[CAD_manager.device_type])
print('\n'+'\033[1m'+'Chip '+CAD_manager.chip+'\033[0m')
area_capa = params.get_params(print_params=True)
number_of_resonators = len(area_capa)


Chip 12
Capacitor capacitance per unit area: c = 7.44 fF/um^2
Number of capacitors: N_capa = 2
Meander length: l_meander = 2.93 mm
Coupling length: l_coupling = 300 um
fr (GHz) | C (fF) | A (um^2) | coupling (um)
11.21 	 | 40 	 | 10.75 	 | 89
10.20 	 | 55 	 | 14.78 	 | 87
9.66 	 | 65 	 | 17.47 	 | 85
8.99 	 | 80 	 | 21.51 	 | 83
8.27 	 | 100 	 | 26.88 	 | 81
7.70 	 | 120 	 | 32.26 	 | 79
7.13 	 | 145 	 | 38.98 	 | 77
6.43 	 | 185 	 | 49.73 	 | 75
5.90 	 | 225 	 | 60.48 	 | 73
5.19 	 | 300 	 | 80.65 	 | 71
4.54 	 | 400 	 | 107.53 	 | 69
4.09 	 | 500 	 | 134.41 	 | 67
3.75 	 | 600 	 | 161.29 	 | 65


In [17]:
# Element design definition
feedline = TWPA_elements_cls.FeedLine()
LER = TWPA_elements_cls.Overlap_LER()
feedline.feedline_length = CAD_manager.grid_size[0]*1000
feedline.feedline_width = device_parameter[CAD_manager.device_type]['feedline_parameters']['width_of_feedline'] 
feedline.feedline_gap = device_parameter[CAD_manager.device_type]['feedline_parameters']['gap_of_feedline'] 
feedline.x_pad = 150 
feedline.y_pad = 300 
feedline.x_arm = 20 
feedline.y_arm = feedline.feedline_width 
feedline.taper_length = 25 
feedline.x_pad_gap = 35 
feedline.y_pad_gap = 175
feedline.arm_gap = feedline.feedline_gap 
feedline.CPW = True
feedline.Exclude_ground = True
LER.resonators_ground_gap = 30
LER.width_of_wire = device_parameter[CAD_manager.device_type]['meander_parameters']['width_of_wire']
LER.spacing_between_steps = device_parameter[CAD_manager.device_type]['meander_parameters']['spacing_between_steps']
LER.length_of_step = device_parameter[CAD_manager.device_type]['meander_parameters']['length_of_step']
LER.length_of_coupling_step = device_parameter[CAD_manager.device_type]['meander_parameters']['length_of_coupling_step']
LER.length_of_meander = device_parameter[CAD_manager.device_type]['meander_parameters']['length_of_meander']
LER.number_of_capacitors = device_parameter[CAD_manager.device_type]['capacitor_parameters']['number_of_capacitors']
LER.spacing_capa = device_parameter[CAD_manager.device_type]['capacitor_parameters']['spacing_between_capa']

In [18]:
# Layer definition
LER.ground_layer = ground_layer
LER.capa_top_layer = capa_top_layer
CAD_manager.pad_top_layer = pad_top_layer
CAD_manager.connection_wire_top_layer = connection_wire_top_layer
CAD_manager.ground_layer = LER.ground_layer
CAD_manager.pad_bottom_layer = LER.bottom_layer
CAD_manager.connection_wire_bottom_layer = LER.bottom_layer
feedline.ground_layer = LER.ground_layer
feedline.feedline_layer = LER.bottom_layer

In [19]:
# CAD manager device info
CAD_manager.resonators_coupling_gap = np.asarray(device_parameter[CAD_manager.device_type]['resonator_parameters']['coupling_distance'])
CAD_manager.resonators = list([] for _ in range(number_of_resonators))
for j in range(0,number_of_resonators):
    LER.area_capa = area_capa[j]
    if j%2 == 0:
        CAD_manager.resonators[j] = LER.generateCell(reflection=True)
    else:
        CAD_manager.resonators[j] = LER.generateCell(reflection=False)
CAD_manager.feedline = feedline
CAD_manager.element = LER
CAD_manager.n_unit_cells = number_of_resonators
CAD_manager.dev_label = CAD_manager.device_type + '_V' + device_version + '_' + CAD_manager.chip
CAD_manager.design_filename = CAD_manager.device_type + '_' + CAD_manager.chip

In [20]:
# gds and job file generation
CAD_manager.generate_GDS()
CAD_manager.populate_dose_matric()
CAD_manager.generate_jobs(row=row, column=column)

## Feedline

In [6]:
importlib.reload(TWPA_parameters_cls)
CAD_manager.load_reset_libs()

In [7]:
dose_matrix = {str(ground_layer):3,
              }

NameError: name 'ground_layer' is not defined

In [ ]:
# Type of device
CAD_manager.device_type = 'feedline'
device_version = '01'

In [ ]:
# Chip
row = 0
column = 0

In [ ]:
### Fab parameters file
dirname = os.path.abspath('')
fab_param_filename = os.path.join(dirname, 'default_parameters\\fab_parameters.json')
with open(fab_param_filename) as json_file:
    fab_parameters = json.load(json_file)

In [ ]:
### Device parameters file
dirname = os.path.abspath('')
device_param_filename = os.path.join(dirname, 'default_parameters\\device_parameters.json')
with open(device_param_filename) as json_file:
    device_parameter = json.load(json_file)

In [8]:
# CAD manager general info
CAD_manager.modulation = False
CAD_manager.chip = str(row)+str(column)
CAD_manager.litho_plan = litho_plan
CAD_manager.dose_matric = dose_matric

NameError: name 'row' is not defined

In [9]:
# Device parameters
params = TWPA_parameters_cls.feedline_parameters(fab_parameters, device_parameter[CAD_manager.device_type])
print('\n'+'\033[1m'+'Chip '+CAD_manager.chip+'\033[0m')

NameError: name 'fab_parameters' is not defined

In [18]:
# Element design definition
feedline = TWPA_elements_cls.FeedLine()
feedline.feedline_length = CAD_manager.grid_size[0]*1000
feedline.feedline_width = device_parameter[CAD_manager.device_type]['feedline_parameters']['width_of_feedline'] 
feedline.feedline_gap = device_parameter[CAD_manager.device_type]['feedline_parameters']['gap_of_feedline'] 
feedline.x_pad = 150 
feedline.y_pad = 300 
feedline.x_arm = 20 
feedline.y_arm = feedline.feedline_width 
feedline.taper_length = 25 
feedline.x_pad_gap = 35 
feedline.y_pad_gap = 175
feedline.arm_gap = feedline.feedline_gap 
feedline.CPW = True
feedline.Exclude_ground = True

In [19]:
# Layer definition
CAD_manager.pad_top_layer = 0
CAD_manager.connection_wire_top_layer = 0
CAD_manager.ground_layer = ground_layer
CAD_manager.pad_bottom_layer = 0
CAD_manager.connection_wire_bottom_layer = 0
feedline.ground_layer = ground_layer
feedline.feedline_layer = 101

In [20]:
# CAD manager device info
CAD_manager.feedline = feedline
CAD_manager.dev_label = CAD_manager.device_type + '_V' + device_version + '_' + CAD_manager.chip
CAD_manager.design_filename = CAD_manager.device_type + '_' + CAD_manager.chip

In [21]:
# gds and job file generation
CAD_manager.generate_GDS()
CAD_manager.populate_dose_matric()
CAD_manager.generate_jobs(row=row, column=column)

In [ ]:
# CAD manager general info
CAD_manager.modulation = False
CAD_manager.chip = str(row)+str(column)
CAD_manager.litho_plan = litho_plan
CAD_manager.dose_matric = dose_matric

# Device parameters
params = TWPA_parameters_cls.feedline_parameters(fab_parameters, device_parameter[CAD_manager.device_type])
print('\n'+'\033[1m'+'Chip '+CAD_manager.chip+'\033[0m')

# Element design definition
feedline = TWPA_elements_cls.FeedLine()
feedline.feedline_length = CAD_manager.grid_size[0]*1000
feedline.feedline_width = device_parameter[CAD_manager.device_type]['feedline_parameters']['width_of_feedline'] 
feedline.feedline_gap = device_parameter[CAD_manager.device_type]['feedline_parameters']['gap_of_feedline'] 
feedline.x_pad = 150 
feedline.y_pad = 300 
feedline.x_arm = 20 
feedline.y_arm = feedline.feedline_width 
feedline.taper_length = 25 
feedline.x_pad_gap = 35 
feedline.y_pad_gap = 175
feedline.arm_gap = feedline.feedline_gap 
feedline.CPW = True
feedline.Exclude_ground = True

# Layer definition
CAD_manager.pad_top_layer = 0
CAD_manager.connection_wire_top_layer = 0
CAD_manager.ground_layer = ground_layer
CAD_manager.pad_bottom_layer = 0
CAD_manager.connection_wire_bottom_layer = 0
feedline.ground_layer = ground_layer
feedline.feedline_layer = 101

# CAD manager device info
CAD_manager.feedline = feedline
CAD_manager.dev_label = CAD_manager.device_type + '_V' + device_version + '_' + CAD_manager.chip
CAD_manager.design_filename = CAD_manager.device_type + '_' + CAD_manager.chip

# gds and job file generation
CAD_manager.generate_GDS()
CAD_manager.populate_dose_matric()
CAD_manager.generate_jobs(row=row, column=column)

## Resistor

In [ ]:
importlib.reload(TWPA_parameters_cls)
CAD_manager.load_reset_libs()

In [ ]:
# Type of device
CAD_manager.device_type = 'CRLH_V04'
device_version = '01'

In [ ]:
# Chip
row = 0
column = 0

In [ ]:
### Fab parameters file
dirname = os.path.abspath('')
fab_param_filename = os.path.join(dirname, 'default_parameters\\fab_parameters.json')
with open(fab_param_filename) as json_file:
    fab_parameters = json.load(json_file)

In [ ]:
### Device parameters file
dirname = os.path.abspath('')
device_param_filename = os.path.join(dirname, 'default_parameters\\device_parameters.json')
with open(device_param_filename) as json_file:
    device_parameter = json.load(json_file)

In [ ]:
# CAD manager general info
CAD_manager.modulation = False
CAD_manager.litho_plan = litho_plan
CAD_manager.dose_matric = dose_matric
CAD_manager.chip = str(row)+str(column)

NameError: name 'litho_plan' is not defined

In [ ]:
# Device parameters
params = TWPA_parameters_cls.CRLH_V04_parameters(fab_parameters, device_parameter[CAD_manager.device_type])
print('\n'+'\033[1m'+'Chip '+CAD_manager.chip+'\033[0m')
params.get_params(print_params=True)


Chip 00
Junction capacitance per unit area: c_j = 45.00 fF/um^2
Junction critical current density: j_c = 90.00 A/cm^2
Capacitor capacitance per unit area: c_c = 1.33 fF/um^2
Number of cells: Ncell = 142
Number of junctions in line per cell: Njj_line = 10
Number of junctions to ground per cell: Njj_ground = 10
Number of capacitors in line per cell: Ncapa_line = 2
Number of capacitors to ground per cell: Ncapa_ground = 2
Junction in line dimensions: H = 5.23 um | W = 1.00 um | A = 5.23 um^2
Room temp resistance in line: Rj = 53.91 Ohm
Critical current in line: Ic = 3.62 uA
Normal state resistance in line: Rn = 91.10 Ohm
Josephson inductance in line: Lj_line = 90.89 pH
Josephson capacitance in line: Cj_line = 235.35 fF
Josephson inductance in line per unit cell: Lj_cell_line = 908.94 pH
Josephson capacitance in line per unit cell: Cj_cell_line = 23.53 fF
Junction to ground dimensions: H = 5.23 um | W = 1.00 um | A = 5.23 um^2
Room temp resistance to ground: Rj = 53.91 Ohm
Critical curren

(5.2299999999999995,
 1.0,
 5.2299999999999995,
 1.0,
 20.0,
 5.379999999999999,
 20.0,
 5.379999999999999)

In [ ]:
# Element design definition
CRLH_V03 = TWPA_elements_cls.CRLH_V03_cell()
CRLH_V03.litho_overlap = 0.1
CRLH_V03.etching_offset = 0.25
CRLH_V03.electrode_height_difference_JJ_line = 0.5
CRLH_V03.electrode_height_difference_capa_line = 0.5
CRLH_V03.electrode_height_difference_JJ_ground = 0.5
CRLH_V03.electrode_height_difference_capa_ground = 0.5
CRLH_V03.y_low_current_ground = 3

CRLH_V03.width_jj_line = device_parameter[CAD_manager.device_type]['junction_parameters']['width_of_junction_in_line'] 
CRLH_V03.height_jj_line = device_parameter[CAD_manager.device_type]['junction_parameters']['height_of_junction_in_line']
CRLH_V03.spacing_jj_line = device_parameter[CAD_manager.device_type]['junction_parameters']['spacing_between_junctions_in_line']
CRLH_V03.bottom_spacing_jj_line = device_parameter[CAD_manager.device_type]['junction_parameters']['bottom_spacing_between_junctions_in_line']
CRLH_V03.top_spacing_jj_line = device_parameter[CAD_manager.device_type]['junction_parameters']['top_spacing_between_junctions_in_line']
CRLH_V03.number_of_junctions_line = device_parameter[CAD_manager.device_type]['junction_parameters']['number_of_junctions_per_unit_cell_in_line']

CRLH_V03.width_capa_line = device_parameter[CAD_manager.device_type]['capacitor_parameters']['width_of_capacitor_in_line']
CRLH_V03.height_capa_line = device_parameter[CAD_manager.device_type]['capacitor_parameters']['height_of_capacitor_in_line']
CRLH_V03.spacing_capa_line = device_parameter[CAD_manager.device_type]['capacitor_parameters']['spacing_between_capacitors_in_line']
CRLH_V03.bottom_spacing_capa_line = device_parameter[CAD_manager.device_type]['capacitor_parameters']['bottom_spacing_between_capacitors_in_line']    
CRLH_V03.top_spacing_capa_line = device_parameter[CAD_manager.device_type]['capacitor_parameters']['top_spacing_between_capacitors_in_line']    
CRLH_V03.number_of_capacitors_line = device_parameter[CAD_manager.device_type]['capacitor_parameters']['number_of_capacitors_per_unit_cell_in_line']

CRLH_V03.width_jj_ground = device_parameter[CAD_manager.device_type]['junction_parameters']['width_of_junction_to_ground'] 
CRLH_V03.height_jj_ground = device_parameter[CAD_manager.device_type]['junction_parameters']['height_of_junction_to_ground']
CRLH_V03.spacing_jj_ground = device_parameter[CAD_manager.device_type]['junction_parameters']['spacing_between_junctions_to_ground']
CRLH_V03.bottom_spacing_jj_ground = device_parameter[CAD_manager.device_type]['junction_parameters']['bottom_spacing_between_junctions_to_ground']
CRLH_V03.top_spacing_jj_ground = device_parameter[CAD_manager.device_type]['junction_parameters']['top_spacing_between_junctions_to_ground']
CRLH_V03.number_of_junctions_ground = device_parameter[CAD_manager.device_type]['junction_parameters']['number_of_junctions_per_unit_cell_to_ground']

CRLH_V03.width_capa_ground = device_parameter[CAD_manager.device_type]['capacitor_parameters']['width_of_capacitor_to_ground']
CRLH_V03.height_capa_ground = device_parameter[CAD_manager.device_type]['capacitor_parameters']['height_of_capacitor_to_ground']
CRLH_V03.spacing_capa_ground = device_parameter[CAD_manager.device_type]['capacitor_parameters']['spacing_between_capacitors_to_ground']
CRLH_V03.bottom_spacing_capa_ground = device_parameter[CAD_manager.device_type]['capacitor_parameters']['bottom_spacing_between_capacitors_to_ground']
CRLH_V03.top_spacing_capa_ground = device_parameter[CAD_manager.device_type]['capacitor_parameters']['top_spacing_between_capacitors_to_ground']
CRLH_V03.number_of_capacitors_ground = device_parameter[CAD_manager.device_type]['capacitor_parameters']['number_of_capacitors_per_unit_cell_to_ground']

In [ ]:
#  Layer definition
CRLH_V03.ground_layer = ground_layer
CAD_manager.pad_bottom_layer = pad_bottom_layer
CAD_manager.connection_wire_bottom_layer = connection_wire_bottom_layer
CRLH_V03.jj_bottom_layer = jj_bottom_layer
CRLH_V03.capa_bottom_layer = capa_bottom_layer
CRLH_V03.jj_top_layer = jj_top_layer
CRLH_V03.capa_top_layer = capa_top_layer
CAD_manager.pad_top_layer = pad_top_layer
CAD_manager.connection_wire_top_layer = connection_wire_top_layer
CAD_manager.ground_layer = CRLH_V03.ground_layer

NameError: name 'ground_layer' is not defined

In [ ]:
# CAD manager device info
CAD_manager.n_unit_cells = device_parameter[CAD_manager.device_type]['TWPA_parameters']['number_of_cells']
CAD_manager.element = CRLH_V03
CAD_manager.dev_label = CAD_manager.device_type + '_V' + device_version + '_' + CAD_manager.chip
CAD_manager.design_filename = CAD_manager.device_type + '_' + CAD_manager.chip

KeyError: 'CRLH_V04'

In [ ]:
# gds and job file generation
CAD_manager.generate_GDS()
CAD_manager.populate_dose_matric()
CAD_manager.generate_jobs(row=row, column=column)

# Job & Batch files generation

In [21]:
CAD_manager.batch_plan = batch_plan
CAD_manager.generate_batch_files()

# Wafer generation

Let's say we now want to write a wafer of 16 chips of the same kind of device varying some parameters from chip to chip.

In [6]:
date = '260415'
WaferName = 'Feedline_test'
WaferNumber = '11'

In [7]:
importlib.reload(CAD_manager_cls)
CAD_manager = CAD_manager_cls.CAD_manager(date, WaferName, WaferNumber)

Junk folder already exists, cleaning it...
Junk folder already exists, cleaning it...


In [8]:
importlib.reload(TWPA_parameters_cls)
CAD_manager.load_reset_libs()

## Layers

In [9]:
bottom_layer = 1
jj_top_layer = 2
capa_top_layer = 3

## Litho plan

In [10]:
litho_plan = {'bottom_layer':[bottom_layer],
              'junctions_top_layer':[jj_top_layer],
              'capacitors_top_layer':[capa_top_layer],
             }

In [11]:
batch_plan = {'bottom_layer':{'jobs':['bottom_layer'],
                              'current':[24],
                              'datum':[8],
                              'sleep':[10]},
              'junctions_top_layer':{'jobs':['junctions_top_layer'],
                              'current':[24],
                              'datum':[8],
                              'sleep':[10]},
              'capa_top_layer':{'jobs':['capa_top_layer'],
                          'current':[24],
                          'datum':[8],
                          'sleep':[10]},
             }

## Doses

In [12]:
dose_matric = {str(bottom_layer):2.2,
               str(jj_top_layer):2.2,
               str(capa_top_layer):2.2,
              }

## Design

Let's add the TWPAs with modulation: both the modulation and the area of the junctions will vary.

### Composite Right-Left-Handed Josephson Transmission Line (V03)

In [14]:
CAD_manager.device_type = 'CRLH_V03'
device_version = '01'

In [15]:
rows = [0, 1, 2, 3]
columns = [0, 1, 2, 3]

In [16]:
Hj_serie_mat = [[4.28, 4.28, 4.28, 4.28],
                [4.52, 4.52, 4.52, 4.52],
                [4.98, 4.98, 4.98, 4.98],
                [5.23, 5.23, 5.23, 5.23]]

Wj_serie_mat = [[1.00, 1.00, 1.00, 1.00],
                [1.00, 1.00, 1.00, 1.00],
                [1.00, 1.00, 1.00, 1.00],
                [1.00, 1.00, 1.00, 1.00]]

Hj_ground_mat = [[4.28, 4.28, 4.28, 4.28],
                 [4.52, 4.52, 4.52, 4.52],
                 [4.98, 4.98, 4.98, 4.98],
                 [5.23, 5.23, 5.23, 5.23]]

Wj_ground_mat = [[1.00, 1.00, 1.00, 1.00],
                 [1.00, 1.00, 1.00, 1.00],
                 [1.00, 1.00, 1.00, 1.00],
                 [1.00, 1.00, 1.00, 1.00]]

Hc_serie_mat = [[20.00, 20.00, 20.00, 20.00],
                [20.00, 20.00, 20.00, 20.00],
                [20.00, 20.00, 20.00, 20.00],
                [20.00, 20.00, 20.00, 20.00]]

Wc_serie_mat = [[5.38, 5.38, 4.03, 4.03],
                [5.38, 5.38, 4.03, 4.03],
                [5.38, 5.38, 4.03, 4.03],
                [5.38, 5.38, 4.03, 4.03]]

Hc_ground_mat = [[20.00, 20.00, 20.00, 20.00],
                 [20.00, 20.00, 20.00, 20.00],
                 [20.00, 20.00, 20.00, 20.00],
                 [20.00, 20.00, 20.00, 20.00]]

Wc_ground_mat = [[5.38, 5.38, 2.69, 2.69],
                 [5.38, 5.38, 2.69, 2.69],
                 [5.38, 5.38, 2.69, 2.69],
                 [5.38, 5.38, 2.69, 2.69]]


TWPA_modulation = [[False, False, False, False],
                   [False, False, False, False],
                   [False, False, False, False],
                   [False, False, False, False]]


In [17]:
### Fab parameters file
dirname = os.path.abspath('')
fab_param_filename = os.path.join(dirname, 'default_parameters\\fab_parameters.json')
with open(fab_param_filename) as json_file:
    fab_parameters = json.load(json_file)

### Device parameters file
dirname = os.path.abspath('')

device_param_filename = os.path.join(dirname, 'default_parameters\\device_parameters.json')
with open(device_param_filename) as json_file:
    device_parameter = json.load(json_file)

In [18]:
for i in range(len(rows)):
    for j in range(len(columns)):
        CAD_manager.load_reset_libs()
        CAD_manager.modulation = TWPA_modulation[i][j]
        CAD_manager.litho_plan = litho_plan
        CAD_manager.dose_matric = dose_matric
        CAD_manager.chip = str(rows[i])+str(columns[j])

        device_parameter[CAD_manager.device_type]['junction_parameters']['height_of_junction_in_line'] = Hj_serie_mat[i][j]
        device_parameter[CAD_manager.device_type]['junction_parameters']['width_of_junction_in_line'] = Wj_serie_mat[i][j]
        device_parameter[CAD_manager.device_type]['junction_parameters']['height_of_junction_to_ground'] = Hj_ground_mat[i][j]
        device_parameter[CAD_manager.device_type]['junction_parameters']['width_of_junction_to_ground'] = Wj_ground_mat[i][j]
        device_parameter[CAD_manager.device_type]['capacitor_parameters']['height_of_capacitor_in_line'] = Hc_serie_mat[i][j]
        device_parameter[CAD_manager.device_type]['capacitor_parameters']['width_of_capacitor_in_line'] = Wc_serie_mat[i][j]
        device_parameter[CAD_manager.device_type]['capacitor_parameters']['height_of_capacitor_to_ground'] = Hc_ground_mat[i][j]
        device_parameter[CAD_manager.device_type]['capacitor_parameters']['width_of_capacitor_to_ground'] = Wc_ground_mat[i][j]
        
        params = TWPA_parameters_cls.CRLH_V03_parameters(fab_parameters, device_parameter[CAD_manager.device_type])
        print('\n'+'\033[1m'+'Chip '+CAD_manager.chip+'\033[0m')
        Hj_serie, Wj_serie, Hj_ground, Wj_ground, Hc_serie, Wc_serie, Hc_ground, Wc_ground = params.get_params(print_params=True, modulation = CAD_manager.modulation)
        
        # Element design definition
        CRLH_V03 = TWPA_elements_cls.CRLH_V03_cell()
        
        CRLH_V03.litho_overlap = 0.1
        CRLH_V03.etching_offset = 0.0
        CRLH_V03.electrode_height_difference_JJ_line = 0.5
        CRLH_V03.electrode_height_difference_capa_line = 0.5
        CRLH_V03.electrode_height_difference_JJ_ground = 0.5
        CRLH_V03.electrode_height_difference_capa_ground = 0.5
        CRLH_V03.y_low_current_ground = 3

        CRLH_V03.width_jj_line = device_parameter[CAD_manager.device_type]['junction_parameters']['width_of_junction_in_line'] 
        CRLH_V03.height_jj_line = device_parameter[CAD_manager.device_type]['junction_parameters']['height_of_junction_in_line']
        CRLH_V03.spacing_jj_line = device_parameter[CAD_manager.device_type]['junction_parameters']['spacing_between_junctions_in_line']
        CRLH_V03.bottom_spacing_jj_line = device_parameter[CAD_manager.device_type]['junction_parameters']['bottom_spacing_between_junctions_in_line']
        CRLH_V03.top_spacing_jj_line = device_parameter[CAD_manager.device_type]['junction_parameters']['top_spacing_between_junctions_in_line']
        CRLH_V03.number_of_junctions_line = device_parameter[CAD_manager.device_type]['junction_parameters']['number_of_junctions_per_unit_cell_in_line']

        CRLH_V03.width_capa_line = device_parameter[CAD_manager.device_type]['capacitor_parameters']['width_of_capacitor_in_line']
        CRLH_V03.height_capa_line = device_parameter[CAD_manager.device_type]['capacitor_parameters']['height_of_capacitor_in_line']
        CRLH_V03.spacing_capa_line = device_parameter[CAD_manager.device_type]['capacitor_parameters']['spacing_between_capacitors_in_line']
        CRLH_V03.bottom_spacing_capa_line = device_parameter[CAD_manager.device_type]['capacitor_parameters']['bottom_spacing_between_capacitors_in_line']    
        CRLH_V03.top_spacing_capa_line = device_parameter[CAD_manager.device_type]['capacitor_parameters']['top_spacing_between_capacitors_in_line']    
        CRLH_V03.number_of_capacitors_line = device_parameter[CAD_manager.device_type]['capacitor_parameters']['number_of_capacitors_per_unit_cell_in_line']

        CRLH_V03.width_jj_ground = device_parameter[CAD_manager.device_type]['junction_parameters']['width_of_junction_to_ground'] 
        CRLH_V03.height_jj_ground = device_parameter[CAD_manager.device_type]['junction_parameters']['height_of_junction_to_ground']
        CRLH_V03.spacing_jj_ground = device_parameter[CAD_manager.device_type]['junction_parameters']['spacing_between_junctions_to_ground']
        CRLH_V03.bottom_spacing_jj_ground = device_parameter[CAD_manager.device_type]['junction_parameters']['bottom_spacing_between_junctions_to_ground']
        CRLH_V03.top_spacing_jj_ground = device_parameter[CAD_manager.device_type]['junction_parameters']['top_spacing_between_junctions_to_ground']
        CRLH_V03.number_of_junctions_ground = device_parameter[CAD_manager.device_type]['junction_parameters']['number_of_junctions_per_unit_cell_to_ground']

        CRLH_V03.width_capa_ground = device_parameter[CAD_manager.device_type]['capacitor_parameters']['width_of_capacitor_to_ground']
        CRLH_V03.height_capa_ground = device_parameter[CAD_manager.device_type]['capacitor_parameters']['height_of_capacitor_to_ground']
        CRLH_V03.spacing_capa_ground = device_parameter[CAD_manager.device_type]['capacitor_parameters']['spacing_between_capacitors_to_ground']
        CRLH_V03.bottom_spacing_capa_ground = device_parameter[CAD_manager.device_type]['capacitor_parameters']['bottom_spacing_between_capacitors_to_ground']
        CRLH_V03.top_spacing_capa_ground = device_parameter[CAD_manager.device_type]['capacitor_parameters']['top_spacing_between_capacitors_to_ground']
        CRLH_V03.number_of_capacitors_ground = device_parameter[CAD_manager.device_type]['capacitor_parameters']['number_of_capacitors_per_unit_cell_to_ground']
                
        # CAD manager device info
        CAD_manager.n_unit_cells = device_parameter[CAD_manager.device_type]['TWPA_parameters']['number_of_cells']
        CAD_manager.element = CRLH_V03
        CAD_manager.dev_label = WaferName + '_V' + device_version + '_Wf' + WaferNumber + '_' + CAD_manager.chip
        CAD_manager.design_filename = WaferName + '_V' + device_version + '_Wf' + WaferNumber + '_' + CAD_manager.chip

        # Layer definition
        CAD_manager.ground_layer = 0
        CAD_manager.pad_bottom_layer = 0
        CAD_manager.connection_wire_bottom_layer = 0
        CAD_manager.pad_top_layer = 0
        CAD_manager.connection_wire_top_layer = 0
        CRLH_V03.bottom_layer = bottom_layer
        CRLH_V03.jj_top_layer = jj_top_layer
        CRLH_V03.capa_top_layer = capa_top_layer
        
        # gds and job file generation
        CAD_manager.generate_GDS()
        CAD_manager.populate_dose_matric()
        CAD_manager.generate_jobs(row=rows[i], column=columns[j])


Chip 00
Junction capacitance per unit area: c_j = 45.00 fF/um^2
Junction critical current density: j_c = 90.00 A/cm^2
Capacitor capacitance per unit area: c_c = 7.44 fF/um^2
Number of cells: Ncell = 108
Number of junctions in line per cell: Njj_line = 10
Number of junctions to ground per cell: Njj_ground = 10
Number of capacitors in line per cell: Ncapa_line = 2
Number of capacitors to ground per cell: Ncapa_ground = 2
Junction in line dimensions: H = 4.28 um | W = 1.00 um | A = 4.28 um^2
Room temp resistance in line: Rj = 65.87 Ohm
Critical current in line: Ic = 2.96 uA
Normal state resistance in line: Rn = 111.33 Ohm
Josephson inductance in line: Lj_line = 111.07 pH
Josephson capacitance in line: Cj_line = 192.60 fF
Josephson inductance in line per unit cell: Lj_cell_line = 1110.69 pH
Josephson capacitance in line per unit cell: Cj_cell_line = 19.26 fF
Junction to ground dimensions: H = 4.28 um | W = 1.00 um | A = 4.28 um^2
Room temp resistance to ground: Rj = 65.87 Ohm
Critical cur

### Composite Right-Left-Handed Josephson Transmission Line (V04)

In [13]:
CAD_manager.device_type = 'CRLH_V04'
device_version = '02'

In [14]:
rows = [0, 1, 2, 3]
columns = [0, 1, 2, 3]

In [15]:
Hj_serie_mat = [[4.28, 4.28, 4.28, 4.28],
                [4.52, 4.52, 4.52, 4.52],
                [4.98, 4.98, 4.98, 4.98],
                [5.23, 5.23, 5.23, 5.23]]

Wj_serie_mat = [[1.00, 1.00, 1.00, 1.00],
                [1.00, 1.00, 1.00, 1.00],
                [1.00, 1.00, 1.00, 1.00],
                [1.00, 1.00, 1.00, 1.00]]

Hj_ground_mat = [[4.28, 4.28, 4.28, 4.28],
                 [4.52, 4.52, 4.52, 4.52],
                 [4.98, 4.98, 4.98, 4.98],
                 [5.23, 5.23, 5.23, 5.23]]

Wj_ground_mat = [[1.00, 1.00, 1.00, 1.00],
                 [1.00, 1.00, 1.00, 1.00],
                 [1.00, 1.00, 1.00, 1.00],
                 [1.00, 1.00, 1.00, 1.00]]

Hc_serie_mat = [[100.00, 100.00, 100.00, 100.00],
                [100.00, 100.00, 100.00, 100.00],
                [100.00, 100.00, 100.00, 100.00],
                [100.00, 100.00, 100.00, 100.00]]

Wc_serie_mat = [[6.02, 6.02, 4.51, 4.51],
                [6.02, 6.02, 4.51, 4.51],
                [6.02, 6.02, 4.51, 4.51],
                [6.02, 6.02, 4.51, 4.51]]

Hc_ground_mat = [[100.00, 100.00, 100.00, 100.00],
                 [100.00, 100.00, 100.00, 100.00],
                 [100.00, 100.00, 100.00, 100.00],
                 [100.00, 100.00, 100.00, 100.00]]

Wc_ground_mat = [[3.01, 3.01, 1.50, 1.50],
                 [3.01, 3.01, 1.50, 1.50],
                 [3.01, 3.01, 1.50, 1.50],
                 [3.01, 3.01, 1.50, 1.50]]


TWPA_modulation = [[False, False, False, False],
                   [False, False, False, False],
                   [False, False, False, False],
                   [False, False, False, False]]

In [16]:
### Fab parameters file
dirname = os.path.abspath('')
fab_param_filename = os.path.join(dirname, 'default_parameters\\fab_parameters.json')
with open(fab_param_filename) as json_file:
    fab_parameters = json.load(json_file)

### Device parameters file
dirname = os.path.abspath('')

device_param_filename = os.path.join(dirname, 'default_parameters\\device_parameters.json')
with open(device_param_filename) as json_file:
    device_parameter = json.load(json_file)

In [17]:
for i in range(len(rows)):
    for j in range(len(columns)):
        CAD_manager.load_reset_libs()
        CAD_manager.modulation = TWPA_modulation[i][j]
        CAD_manager.litho_plan = litho_plan
        CAD_manager.dose_matric = dose_matric
        CAD_manager.chip = str(rows[i])+str(columns[j])

        device_parameter[CAD_manager.device_type]['junction_parameters']['height_of_junction_in_line'] = Hj_serie_mat[i][j]
        device_parameter[CAD_manager.device_type]['junction_parameters']['width_of_junction_in_line'] = Wj_serie_mat[i][j]
        device_parameter[CAD_manager.device_type]['junction_parameters']['height_of_junction_to_ground'] = Hj_ground_mat[i][j]
        device_parameter[CAD_manager.device_type]['junction_parameters']['width_of_junction_to_ground'] = Wj_ground_mat[i][j]
        device_parameter[CAD_manager.device_type]['capacitor_parameters']['height_of_capacitor_in_line'] = Hc_serie_mat[i][j]
        device_parameter[CAD_manager.device_type]['capacitor_parameters']['width_of_capacitor_in_line'] = Wc_serie_mat[i][j]
        device_parameter[CAD_manager.device_type]['capacitor_parameters']['height_of_capacitor_to_ground'] = Hc_ground_mat[i][j]
        device_parameter[CAD_manager.device_type]['capacitor_parameters']['width_of_capacitor_to_ground'] = Wc_ground_mat[i][j]
        
        params = TWPA_parameters_cls.CRLH_V04_parameters(fab_parameters, device_parameter[CAD_manager.device_type])
        print('\n'+'\033[1m'+'Chip '+CAD_manager.chip+'\033[0m')
        Hj_serie, Wj_serie, Hj_ground, Wj_ground, Hc_serie, Wc_serie, Hc_ground, Wc_ground = params.get_params(print_params=True, modulation = CAD_manager.modulation)
        
        # Element design definition
        CRLH_V04 = TWPA_elements_cls.CRLH_V04_cell()
        
        CRLH_V04.litho_overlap = 0.1
        CRLH_V04.etching_offset = 0.0
        CRLH_V04.electrode_height_difference_JJ_line = 0.5
        CRLH_V04.electrode_height_difference_capa_line = 0.5
        CRLH_V04.electrode_height_difference_JJ_ground = 0.5
        CRLH_V04.electrode_height_difference_capa_ground = 0.5
        CRLH_V04.y_low_current_ground = 3

        CRLH_V04.width_jj_line = device_parameter[CAD_manager.device_type]['junction_parameters']['width_of_junction_in_line'] 
        CRLH_V04.height_jj_line = device_parameter[CAD_manager.device_type]['junction_parameters']['height_of_junction_in_line']
        CRLH_V04.spacing_jj_line = device_parameter[CAD_manager.device_type]['junction_parameters']['spacing_between_junctions_in_line']
        CRLH_V04.bottom_spacing_jj_line = device_parameter[CAD_manager.device_type]['junction_parameters']['bottom_spacing_between_junctions_in_line']
        CRLH_V04.top_spacing_jj_line = device_parameter[CAD_manager.device_type]['junction_parameters']['top_spacing_between_junctions_in_line']
        CRLH_V04.number_of_junctions_line = device_parameter[CAD_manager.device_type]['junction_parameters']['number_of_junctions_per_unit_cell_in_line']

        CRLH_V04.width_capa_line = device_parameter[CAD_manager.device_type]['capacitor_parameters']['width_of_capacitor_in_line']
        CRLH_V04.height_capa_line = device_parameter[CAD_manager.device_type]['capacitor_parameters']['height_of_capacitor_in_line']
        CRLH_V04.spacing_capa_line = device_parameter[CAD_manager.device_type]['capacitor_parameters']['spacing_between_capacitors_in_line']
        CRLH_V04.bottom_spacing_capa_line = device_parameter[CAD_manager.device_type]['capacitor_parameters']['bottom_spacing_between_capacitors_in_line']    
        CRLH_V04.top_spacing_capa_line = device_parameter[CAD_manager.device_type]['capacitor_parameters']['top_spacing_between_capacitors_in_line']    
        CRLH_V04.number_of_capacitors_line = device_parameter[CAD_manager.device_type]['capacitor_parameters']['number_of_capacitors_per_unit_cell_in_line']

        CRLH_V04.width_jj_ground = device_parameter[CAD_manager.device_type]['junction_parameters']['width_of_junction_to_ground'] 
        CRLH_V04.height_jj_ground = device_parameter[CAD_manager.device_type]['junction_parameters']['height_of_junction_to_ground']
        CRLH_V04.spacing_jj_ground = device_parameter[CAD_manager.device_type]['junction_parameters']['spacing_between_junctions_to_ground']
        CRLH_V04.bottom_spacing_jj_ground = device_parameter[CAD_manager.device_type]['junction_parameters']['bottom_spacing_between_junctions_to_ground']
        CRLH_V04.top_spacing_jj_ground = device_parameter[CAD_manager.device_type]['junction_parameters']['top_spacing_between_junctions_to_ground']
        CRLH_V04.number_of_junctions_ground = device_parameter[CAD_manager.device_type]['junction_parameters']['number_of_junctions_per_unit_cell_to_ground']

        CRLH_V04.width_capa_ground = device_parameter[CAD_manager.device_type]['capacitor_parameters']['width_of_capacitor_to_ground']
        CRLH_V04.height_capa_ground = device_parameter[CAD_manager.device_type]['capacitor_parameters']['height_of_capacitor_to_ground']
        CRLH_V04.spacing_capa_ground = device_parameter[CAD_manager.device_type]['capacitor_parameters']['spacing_between_capacitors_to_ground']
        CRLH_V04.bottom_spacing_capa_ground = device_parameter[CAD_manager.device_type]['capacitor_parameters']['bottom_spacing_between_capacitors_to_ground']
        CRLH_V04.top_spacing_capa_ground = device_parameter[CAD_manager.device_type]['capacitor_parameters']['top_spacing_between_capacitors_to_ground']
        CRLH_V04.number_of_capacitors_ground = device_parameter[CAD_manager.device_type]['capacitor_parameters']['number_of_capacitors_per_unit_cell_to_ground']
                
        # CAD manager device info
        CAD_manager.n_unit_cells = device_parameter[CAD_manager.device_type]['TWPA_parameters']['number_of_cells']
        CAD_manager.element = CRLH_V04
        CAD_manager.dev_label = WaferName + '_V' + device_version + '_Wf' + WaferNumber + '_' + CAD_manager.chip
        CAD_manager.design_filename = WaferName + '_V' + device_version + '_Wf' + WaferNumber + '_' + CAD_manager.chip

        # Layer definition
        CAD_manager.ground_layer = 0
        CAD_manager.pad_bottom_layer = 0
        CAD_manager.connection_wire_bottom_layer = 0
        CAD_manager.pad_top_layer = 0
        CAD_manager.connection_wire_top_layer = 0
        CRLH_V04.bottom_layer = bottom_layer
        CRLH_V04.jj_top_layer = jj_top_layer
        CRLH_V04.capa_top_layer = capa_top_layer
        
        # gds and job file generation
        CAD_manager.generate_GDS()
        CAD_manager.populate_dose_matric()
        CAD_manager.generate_jobs(row=rows[i], column=columns[j])


Chip 00
Junction capacitance per unit area: c_j = 45.00 fF/um^2
Junction critical current density: j_c = 90.00 A/cm^2
Capacitor capacitance per unit area: c_c = 1.33 fF/um^2
Number of cells: Ncell = 116
Number of junctions in line per cell: Njj_line = 10
Number of junctions to ground per cell: Njj_ground = 10
Number of capacitors in line per cell: Ncapa_line = 2
Number of capacitors to ground per cell: Ncapa_ground = 2
Junction in line dimensions: H = 4.28 um | W = 1.00 um | A = 4.28 um^2
Room temp resistance in line: Rj = 65.87 Ohm
Critical current in line: Ic = 2.96 uA
Normal state resistance in line: Rn = 111.33 Ohm
Josephson inductance in line: Lj_line = 111.07 pH
Josephson capacitance in line: Cj_line = 192.60 fF
Josephson inductance in line per unit cell: Lj_cell_line = 1110.69 pH
Josephson capacitance in line per unit cell: Cj_cell_line = 19.26 fF
Junction to ground dimensions: H = 4.28 um | W = 1.00 um | A = 4.28 um^2
Room temp resistance to ground: Rj = 65.87 Ohm
Critical cur

### Composite Right-Left-Handed Josephson Transmission Line (V02)

In [18]:
CAD_manager.device_type = 'CRLH_V02'
device_version = '02'

In [ ]:
rows = [0, 1, 2, 3]
columns = [0, 1, 2, 3]

In [ ]:
Hj_serie_mat = [[4.28, 4.28, 4.28, 4.28],
                [4.52, 4.52, 4.52, 4.52],
                [4.98, 4.98, 4.98, 4.98],
                [5.23, 5.23, 5.23, 5.23]]

Wj_serie_mat = [[1.00, 1.00, 1.00, 1.00],
                [1.00, 1.00, 1.00, 1.00],
                [1.00, 1.00, 1.00, 1.00],
                [1.00, 1.00, 1.00, 1.00]]

Hj_ground_mat = [[4.28, 4.28, 4.28, 4.28],
                 [4.52, 4.52, 4.52, 4.52],
                 [4.98, 4.98, 4.98, 4.98],
                 [5.23, 5.23, 5.23, 5.23]]

Wj_ground_mat = [[1.00, 1.00, 1.00, 1.00],
                 [1.00, 1.00, 1.00, 1.00],
                 [1.00, 1.00, 1.00, 1.00],
                 [1.00, 1.00, 1.00, 1.00]]

Hc_serie_mat = [[100.00, 100.00, 100.00, 100.00],
                [100.00, 100.00, 100.00, 100.00],
                [100.00, 100.00, 100.00, 100.00],
                [100.00, 100.00, 100.00, 100.00]]

Wc_serie_mat = [[6.02, 6.02, 4.51, 4.51],
                [6.02, 6.02, 4.51, 4.51],
                [6.02, 6.02, 4.51, 4.51],
                [6.02, 6.02, 4.51, 4.51]]

Hc_ground_mat = [[100.00, 100.00, 100.00, 100.00],
                 [100.00, 100.00, 100.00, 100.00],
                 [100.00, 100.00, 100.00, 100.00],
                 [100.00, 100.00, 100.00, 100.00]]

Wc_ground_mat = [[3.01, 3.01, 1.50, 1.50],
                 [3.01, 3.01, 1.50, 1.50],
                 [3.01, 3.01, 1.50, 1.50],
                 [3.01, 3.01, 1.50, 1.50]]


TWPA_modulation = [[False, False, False, False],
                   [False, False, False, False],
                   [False, False, False, False],
                   [False, False, False, False]]

In [2]:
### Fab parameters file
dirname = os.path.abspath('')
fab_param_filename = os.path.join(dirname, 'default_parameters\\fab_parameters.json')
with open(fab_param_filename) as json_file:
    fab_parameters = json.load(json_file)

### Device parameters file
dirname = os.path.abspath('')

device_param_filename = os.path.join(dirname, 'default_parameters\\device_parameters.json')
with open(device_param_filename) as json_file:
    device_parameter = json.load(json_file)

NameError: name 'os' is not defined

In [3]:
for i in range(len(rows)):
    for j in range(len(columns)):
        CAD_manager.load_reset_libs()
        CAD_manager.modulation = TWPA_modulation[i][j]
        CAD_manager.litho_plan = litho_plan
        CAD_manager.dose_matric = dose_matric
        CAD_manager.chip = str(rows[i])+str(columns[j])

        device_parameter[CAD_manager.device_type]['junction_parameters']['height_of_junction_in_line'] = Hj_serie_mat[i][j]
        device_parameter[CAD_manager.device_type]['junction_parameters']['width_of_junction_in_line'] = Wj_serie_mat[i][j]
        device_parameter[CAD_manager.device_type]['junction_parameters']['height_of_junction_to_ground'] = Hj_ground_mat[i][j]
        device_parameter[CAD_manager.device_type]['junction_parameters']['width_of_junction_to_ground'] = Wj_ground_mat[i][j]
        device_parameter[CAD_manager.device_type]['capacitor_parameters']['height_of_capacitor_in_line'] = Hc_serie_mat[i][j]
        device_parameter[CAD_manager.device_type]['capacitor_parameters']['width_of_capacitor_in_line'] = Wc_serie_mat[i][j]
        device_parameter[CAD_manager.device_type]['capacitor_parameters']['height_of_capacitor_to_ground'] = Hc_ground_mat[i][j]
        device_parameter[CAD_manager.device_type]['capacitor_parameters']['width_of_capacitor_to_ground'] = Wc_ground_mat[i][j]
        
        params = TWPA_parameters_cls.CRLH_V02_parameters(fab_parameters, device_parameter[CAD_manager.device_type])
        print('\n'+'\033[1m'+'Chip '+CAD_manager.chip+'\033[0m')
        Hj_serie, Wj_serie, Hj_ground, Wj_ground, Hc_serie, Wc_serie, Hc_ground, Wc_ground = params.get_params(print_params=True, modulation = CAD_manager.modulation)
        
        # Element design definition
        CRLH_V02 = TWPA_elements_cls.CRLH_V02_cell()
        
        CRLH_V02.litho_overlap = 0.1
        CRLH_V02.etching_offset = 0.0
        CRLH_V02.electrode_height_difference_JJ_line = 0.5
        CRLH_V02.electrode_height_difference_capa_line = 0.5
        CRLH_V02.electrode_height_difference_JJ_ground = 0.5
        CRLH_V02.electrode_height_difference_capa_ground = 0.5
        CRLH_V02.y_low_current_ground = 3

        CRLH_V02.width_jj_line = device_parameter[CAD_manager.device_type]['junction_parameters']['width_of_junction_in_line'] 
        CRLH_V02.height_jj_line = device_parameter[CAD_manager.device_type]['junction_parameters']['height_of_junction_in_line']
        CRLH_V02.spacing_jj_line = device_parameter[CAD_manager.device_type]['junction_parameters']['spacing_between_junctions_in_line']
        CRLH_V02.bottom_spacing_jj_line = device_parameter[CAD_manager.device_type]['junction_parameters']['bottom_spacing_between_junctions_in_line']
        CRLH_V02.top_spacing_jj_line = device_parameter[CAD_manager.device_type]['junction_parameters']['top_spacing_between_junctions_in_line']
        CRLH_V02.number_of_junctions_line = device_parameter[CAD_manager.device_type]['junction_parameters']['number_of_junctions_per_unit_cell_in_line']

        CRLH_V02.width_capa_line = device_parameter[CAD_manager.device_type]['capacitor_parameters']['width_of_capacitor_in_line']
        CRLH_V02.height_capa_line = device_parameter[CAD_manager.device_type]['capacitor_parameters']['height_of_capacitor_in_line']
        CRLH_V02.spacing_capa_line = device_parameter[CAD_manager.device_type]['capacitor_parameters']['spacing_between_capacitors_in_line']
        CRLH_V02.bottom_spacing_capa_line = device_parameter[CAD_manager.device_type]['capacitor_parameters']['bottom_spacing_between_capacitors_in_line']    
        CRLH_V02.top_spacing_capa_line = device_parameter[CAD_manager.device_type]['capacitor_parameters']['top_spacing_between_capacitors_in_line']    
        CRLH_V02.number_of_capacitors_line = device_parameter[CAD_manager.device_type]['capacitor_parameters']['number_of_capacitors_per_unit_cell_in_line']

        CRLH_V02.width_jj_ground = device_parameter[CAD_manager.device_type]['junction_parameters']['width_of_junction_to_ground'] 
        CRLH_V02.height_jj_ground = device_parameter[CAD_manager.device_type]['junction_parameters']['height_of_junction_to_ground']
        CRLH_V02.spacing_jj_ground = device_parameter[CAD_manager.device_type]['junction_parameters']['spacing_between_junctions_to_ground']
        CRLH_V02.bottom_spacing_jj_ground = device_parameter[CAD_manager.device_type]['junction_parameters']['bottom_spacing_between_junctions_to_ground']
        CRLH_V02.top_spacing_jj_ground = device_parameter[CAD_manager.device_type]['junction_parameters']['top_spacing_between_junctions_to_ground']
        CRLH_V02.number_of_junctions_ground = device_parameter[CAD_manager.device_type]['junction_parameters']['number_of_junctions_per_unit_cell_to_ground']

        CRLH_V02.width_capa_ground = device_parameter[CAD_manager.device_type]['capacitor_parameters']['width_of_capacitor_to_ground']
        CRLH_V02.height_capa_ground = device_parameter[CAD_manager.device_type]['capacitor_parameters']['height_of_capacitor_to_ground']
        CRLH_V02.spacing_capa_ground = device_parameter[CAD_manager.device_type]['capacitor_parameters']['spacing_between_capacitors_to_ground']
        CRLH_V02.bottom_spacing_capa_ground = device_parameter[CAD_manager.device_type]['capacitor_parameters']['bottom_spacing_between_capacitors_to_ground']
        CRLH_V02.top_spacing_capa_ground = device_parameter[CAD_manager.device_type]['capacitor_parameters']['top_spacing_between_capacitors_to_ground']
        CRLH_V02.number_of_capacitors_ground = device_parameter[CAD_manager.device_type]['capacitor_parameters']['number_of_capacitors_per_unit_cell_to_ground']
                
        # CAD manager device info
        CAD_manager.n_unit_cells = device_parameter[CAD_manager.device_type]['TWPA_parameters']['number_of_cells']
        CAD_manager.element = CRLH_V02
        CAD_manager.dev_label = WaferName + '_V' + device_version + '_Wf' + WaferNumber + '_' + CAD_manager.chip
        CAD_manager.design_filename = WaferName + '_V' + device_version + '_Wf' + WaferNumber + '_' + CAD_manager.chip

        # Layer definition
        CRLH_V02.ground_layer = ground_layer
        CAD_manager.pad_bottom_layer = pad_bottom_layer
        CAD_manager.connection_wire_bottom_layer = connection_wire_bottom_layer
        CRLH_V02.jj_bottom_layer = jj_bottom_layer
        CRLH_V02.capa_bottom_layer = capa_bottom_layer
        CRLH_V02.jj_top_layer = jj_top_layer
        CRLH_V02.capa_top_layer = capa_top_layer
        CAD_manager.pad_top_layer = pad_top_layer
        CAD_manager.connection_wire_top_layer = connection_wire_top_layer
        CAD_manager.ground_layer = CRLH_V02.ground_layer
        
        # gds and job file generation
        CAD_manager.generate_GDS()
        CAD_manager.populate_dose_matric()
        CAD_manager.generate_jobs(row=rows[i], column=columns[j])

NameError: name 'rows' is not defined

### Composite Right-Left-Handed Josephson Transmission Line (V01)

In [24]:
ground_layer = 1
pad_bottom_layer = 2
connection_wire_bottom_layer = 3
jj_bottom_layer = 4
capa_bottom_layer = 5
jj_top_layer = 6
capa_top_layer = 7
pad_top_layer = 8
connection_wire_top_layer = 9

In [25]:
litho_plan = {'ground':[ground_layer],
              'pad_bottom_layer':[pad_bottom_layer,connection_wire_bottom_layer],
              'bottom_layer':[jj_bottom_layer,capa_bottom_layer],
              'junctions_top_layer':[jj_top_layer],
              'capacitors_top_layer':[capa_top_layer],
             }

In [26]:
batch_plan = {'ground':{'jobs':['ground'],                                                                                                               
                        'current':[15],
                        'datum':[8],
                        'sleep':[10]},
              'bottom_layer':{'jobs':['bottom_layer','pad_bottom_layer'],
                              'current':[12,12],
                              'datum':[8,8],
                              'sleep':[10,20]},
              'junctions_top_layer':{'jobs':['junctions_top_layer'],
                              'current':[3.6],
                              'datum':[8,8],
                              'sleep':[10,20]},
              'capa_top_layer':{'jobs':['capa_top_layer'],
                          'current':[3.6],
                          'datum':[8,8],
                          'sleep':[10,20]},
             }

In [27]:
dose_matric = {str(ground_layer):3,
               str(pad_bottom_layer):18,
               str(connection_wire_bottom_layer):18,
               str(jj_bottom_layer):18,
               str(capa_bottom_layer):16,
               str(jj_top_layer):16,
               str(capa_top_layer):16,
              }

In [28]:
CAD_manager.device_type = 'CRLH_V01'
device_version = '01'

In [29]:
rows = [0, 1, 2, 3]
columns = [0, 1, 2, 3]

In [30]:
Hj_serie_mat = [[4.28, 4.28, 4.28, 4.28],
                [4.52, 4.52, 4.52, 4.52],
                [4.98, 4.98, 4.98, 4.98],
                [5.23, 5.23, 5.23, 5.23]]

Wj_serie_mat = [[1.00, 1.00, 1.00, 1.00],
                [1.00, 1.00, 1.00, 1.00],
                [1.00, 1.00, 1.00, 1.00],
                [1.00, 1.00, 1.00, 1.00]]

Hj_ground_mat = [[4.28, 4.28, 4.28, 4.28],
                 [4.52, 4.52, 4.52, 4.52],
                 [4.98, 4.98, 4.98, 4.98],
                 [5.23, 5.23, 5.23, 5.23]]

Wj_ground_mat = [[1.00, 1.00, 1.00, 1.00],
                 [1.00, 1.00, 1.00, 1.00],
                 [1.00, 1.00, 1.00, 1.00],
                 [1.00, 1.00, 1.00, 1.00]]

Hc_serie_mat = [[20.00, 20.00, 20.00, 20.00],
                [20.00, 20.00, 20.00, 20.00],
                [20.00, 20.00, 20.00, 20.00],
                [20.00, 20.00, 20.00, 20.00]]

Wc_serie_mat = [[5.38, 5.38, 4.03, 4.03],
                [5.38, 5.38, 4.03, 4.03],
                [5.38, 5.38, 4.03, 4.03],
                [5.38, 5.38, 4.03, 4.03]]

Hc_ground_mat = [[20.00, 20.00, 20.00, 20.00],
                 [20.00, 20.00, 20.00, 20.00],
                 [20.00, 20.00, 20.00, 20.00],
                 [20.00, 20.00, 20.00, 20.00]]

Wc_ground_mat = [[5.38, 5.38, 2.69, 2.69],
                 [5.38, 5.38, 2.69, 2.69],
                 [5.38, 5.38, 2.69, 2.69],
                 [5.38, 5.38, 2.69, 2.69]]


TWPA_modulation = [[False, False, False, False],
                   [False, False, False, False],
                   [False, False, False, False],
                   [False, False, False, False]]


In [31]:
### Fab parameters file
dirname = os.path.abspath('')
fab_param_filename = os.path.join(dirname, 'default_parameters\\fab_parameters.json')
with open(fab_param_filename) as json_file:
    fab_parameters = json.load(json_file)

### Device parameters file
dirname = os.path.abspath('')

device_param_filename = os.path.join(dirname, 'default_parameters\\device_parameters.json')
with open(device_param_filename) as json_file:
    device_parameter = json.load(json_file)

In [32]:
for i in range(len(rows)):
    for j in range(len(columns)):
        CAD_manager.load_reset_libs()
        CAD_manager.modulation = TWPA_modulation[i][j]
        CAD_manager.litho_plan = litho_plan
        CAD_manager.dose_matric = dose_matric
        CAD_manager.chip = str(rows[i])+str(columns[j])

        device_parameter[CAD_manager.device_type]['junction_parameters']['height_of_junction_in_line'] = Hj_serie_mat[i][j]
        device_parameter[CAD_manager.device_type]['junction_parameters']['width_of_junction_in_line'] = Wj_serie_mat[i][j]
        device_parameter[CAD_manager.device_type]['junction_parameters']['height_of_junction_to_ground'] = Hj_ground_mat[i][j]
        device_parameter[CAD_manager.device_type]['junction_parameters']['width_of_junction_to_ground'] = Wj_ground_mat[i][j]
        device_parameter[CAD_manager.device_type]['capacitor_parameters']['height_of_capacitor_in_line'] = Hc_serie_mat[i][j]
        device_parameter[CAD_manager.device_type]['capacitor_parameters']['width_of_capacitor_in_line'] = Wc_serie_mat[i][j]
        device_parameter[CAD_manager.device_type]['capacitor_parameters']['height_of_capacitor_to_ground'] = Hc_ground_mat[i][j]
        device_parameter[CAD_manager.device_type]['capacitor_parameters']['width_of_capacitor_to_ground'] = Wc_ground_mat[i][j]
        
        params = TWPA_parameters_cls.CRLH_V01_parameters(fab_parameters, device_parameter[CAD_manager.device_type])
        print('\n'+'\033[1m'+'Chip '+CAD_manager.chip+'\033[0m')
        Hj_serie, Wj_serie, Hj_ground, Wj_ground, Hc_serie, Wc_serie, Hc_ground, Wc_ground = params.get_params(print_params=True, modulation = CAD_manager.modulation)
        
        # Element design definition
        CRLH_V01 = TWPA_elements_cls.CRLH_V01_cell()
        
        CRLH_V01.litho_overlap = 0.1
        CRLH_V01.etching_offset = 0.0
        CRLH_V01.electrode_height_difference_JJ_line = 0.5
        CRLH_V01.electrode_height_difference_capa_line = 0.5
        CRLH_V01.electrode_height_difference_JJ_ground = 0.5
        CRLH_V01.electrode_height_difference_capa_ground = 0.5
        CRLH_V01.y_low_current_ground = 3

        CRLH_V01.width_jj_line = device_parameter[CAD_manager.device_type]['junction_parameters']['width_of_junction_in_line'] 
        CRLH_V01.height_jj_line = device_parameter[CAD_manager.device_type]['junction_parameters']['height_of_junction_in_line']
        CRLH_V01.spacing_jj_line = device_parameter[CAD_manager.device_type]['junction_parameters']['spacing_between_junctions_in_line']
        CRLH_V01.bottom_spacing_jj_line = device_parameter[CAD_manager.device_type]['junction_parameters']['bottom_spacing_between_junctions_in_line']
        CRLH_V01.top_spacing_jj_line = device_parameter[CAD_manager.device_type]['junction_parameters']['top_spacing_between_junctions_in_line']
        CRLH_V01.number_of_junctions_line = device_parameter[CAD_manager.device_type]['junction_parameters']['number_of_junctions_per_unit_cell_in_line']

        CRLH_V01.width_capa_line = device_parameter[CAD_manager.device_type]['capacitor_parameters']['width_of_capacitor_in_line']
        CRLH_V01.height_capa_line = device_parameter[CAD_manager.device_type]['capacitor_parameters']['height_of_capacitor_in_line']
        CRLH_V01.spacing_capa_line = device_parameter[CAD_manager.device_type]['capacitor_parameters']['spacing_between_capacitors_in_line']
        CRLH_V01.bottom_spacing_capa_line = device_parameter[CAD_manager.device_type]['capacitor_parameters']['bottom_spacing_between_capacitors_in_line']    
        CRLH_V01.top_spacing_capa_line = device_parameter[CAD_manager.device_type]['capacitor_parameters']['top_spacing_between_capacitors_in_line']    
        CRLH_V01.number_of_capacitors_line = device_parameter[CAD_manager.device_type]['capacitor_parameters']['number_of_capacitors_per_unit_cell_in_line']

        CRLH_V01.width_jj_ground = device_parameter[CAD_manager.device_type]['junction_parameters']['width_of_junction_to_ground'] 
        CRLH_V01.height_jj_ground = device_parameter[CAD_manager.device_type]['junction_parameters']['height_of_junction_to_ground']
        CRLH_V01.spacing_jj_ground = device_parameter[CAD_manager.device_type]['junction_parameters']['spacing_between_junctions_to_ground']
        CRLH_V01.bottom_spacing_jj_ground = device_parameter[CAD_manager.device_type]['junction_parameters']['bottom_spacing_between_junctions_to_ground']
        CRLH_V01.top_spacing_jj_ground = device_parameter[CAD_manager.device_type]['junction_parameters']['top_spacing_between_junctions_to_ground']
        CRLH_V01.number_of_junctions_ground = device_parameter[CAD_manager.device_type]['junction_parameters']['number_of_junctions_per_unit_cell_to_ground']

        CRLH_V01.width_capa_ground = device_parameter[CAD_manager.device_type]['capacitor_parameters']['width_of_capacitor_to_ground']
        CRLH_V01.height_capa_ground = device_parameter[CAD_manager.device_type]['capacitor_parameters']['height_of_capacitor_to_ground']
        CRLH_V01.spacing_capa_ground = device_parameter[CAD_manager.device_type]['capacitor_parameters']['spacing_between_capacitors_to_ground']
        CRLH_V01.bottom_spacing_capa_ground = device_parameter[CAD_manager.device_type]['capacitor_parameters']['bottom_spacing_between_capacitors_to_ground']
        CRLH_V01.top_spacing_capa_ground = device_parameter[CAD_manager.device_type]['capacitor_parameters']['top_spacing_between_capacitors_to_ground']
        CRLH_V01.number_of_capacitors_ground = device_parameter[CAD_manager.device_type]['capacitor_parameters']['number_of_capacitors_per_unit_cell_to_ground']
                
        # CAD manager device info
        CAD_manager.n_unit_cells = device_parameter[CAD_manager.device_type]['TWPA_parameters']['number_of_cells']
        CAD_manager.element = CRLH_V01
        CAD_manager.dev_label = WaferName + '_V' + device_version + '_Wf' + WaferNumber + '_' + CAD_manager.chip
        CAD_manager.design_filename = WaferName + '_V' + device_version + '_Wf' + WaferNumber + '_' + CAD_manager.chip

        # Layer definition
        CRLH_V01.ground_layer = ground_layer
        CAD_manager.pad_bottom_layer = pad_bottom_layer
        CAD_manager.connection_wire_bottom_layer = connection_wire_bottom_layer
        CRLH_V01.jj_bottom_layer = jj_bottom_layer
        CRLH_V01.capa_bottom_layer = capa_bottom_layer
        CRLH_V01.jj_top_layer = jj_top_layer
        CRLH_V01.capa_top_layer = capa_top_layer
        CAD_manager.pad_top_layer = pad_top_layer
        CAD_manager.connection_wire_top_layer = connection_wire_top_layer
        CAD_manager.ground_layer = CRLH_V01.ground_layer
        
        # gds and job file generation
        CAD_manager.generate_GDS()
        CAD_manager.populate_dose_matric()
        CAD_manager.generate_jobs(row=rows[i], column=columns[j])


Chip 00
Junction capacitance per unit area: c_j = 45.00 fF/um^2
Junction critical current density: j_c = 90.00 A/cm^2
Capacitor capacitance per unit area: c_c = 7.44 fF/um^2
Number of cells: Ncell = 128
Number of junctions in line per cell: Njj_line = 10
Number of junctions to ground per cell: Njj_ground = 10
Number of capacitors in line per cell: Ncapa_line = 2
Number of capacitors to ground per cell: Ncapa_ground = 2
Junction in line dimensions: H = 4.28 um | W = 1.00 um | A = 4.28 um^2
Room temp resistance in line: Rj = 65.87 Ohm
Critical current in line: Ic = 2.96 uA
Normal state resistance in line: Rn = 111.33 Ohm
Josephson inductance in line: Lj_line = 111.07 pH
Josephson capacitance in line: Cj_line = 192.60 fF
Josephson inductance in line per unit cell: Lj_cell_line = 1110.69 pH
Josephson capacitance in line per unit cell: Cj_cell_line = 19.26 fF
Junction to ground dimensions: H = 4.28 um | W = 1.00 um | A = 4.28 um^2
Room temp resistance to ground: Rj = 65.87 Ohm
Critical cur

### Feedline

In [9]:
ground_layer = 1
feedline_layer = 101

In [10]:
litho_plan = {'ground':[ground_layer],
             }

In [11]:
batch_plan = {'ground':{'jobs':['ground'],                                                                                                               
                        'current':[15],
                        'datum':[8],
                        'sleep':[10]},
             }

In [12]:
dose_matric = {str(ground_layer):3,
              }

In [13]:
CAD_manager.device_type = 'feedline'
device_version = '01'

In [14]:
rows = [0, 1, 2, 3]
columns = [0, 1, 2, 3]

In [15]:
feedline_width_mat = [[1177, 1177, 1177, 1177],
                      [1177, 1177, 1177, 1177],
                      [1177, 1177, 1177, 1177],
                      [1177, 1177, 1177, 1177]]

feedline_gap_mat = [[1.00, 1.00, 1.00, 1.00],
                    [1.00, 1.00, 1.00, 1.00],
                    [1.00, 1.00, 1.00, 1.00],
                    [1.00, 1.00, 1.00, 1.00]]

In [16]:
### Fab parameters file
dirname = os.path.abspath('')
fab_param_filename = os.path.join(dirname, 'default_parameters\\fab_parameters.json')
with open(fab_param_filename) as json_file:
    fab_parameters = json.load(json_file)

### Device parameters file
dirname = os.path.abspath('')

device_param_filename = os.path.join(dirname, 'default_parameters\\device_parameters.json')
with open(device_param_filename) as json_file:
    device_parameter = json.load(json_file)

In [ ]:
for i in range(len(rows)):
    for j in range(len(columns)):
        CAD_manager.load_reset_libs()
        CAD_manager.litho_plan = litho_plan
        CAD_manager.dose_matric = dose_matric
        CAD_manager.chip = str(rows[i])+str(columns[j])

        device_parameter[CAD_manager.device_type]['feedline_parameters']['width_of_feedline'] = feedline_width_mat[i][j]
        device_parameter[CAD_manager.device_type]['feedline_parameters']['gap_of_feedline'] = feedline_gap_mat[i][j]

        # Element design definition
        feedline = TWPA_elements_cls.FeedLine()
        feedline.feedline_length = CAD_manager.grid_size[0]*1000 
        feedline.feedline_width = device_parameter[CAD_manager.device_type]['feedline_parameters']['width_of_feedline'] 
        feedline.feedline_gap = device_parameter[CAD_manager.device_type]['feedline_parameters']['gap_of_feedline'] 
        feedline.x_pad = 150
        feedline.y_pad = 300
        feedline.x_arm = 20
        feedline.y_arm = feedline.feedline_width 
        feedline.taper_length = 25
        feedline.x_pad_gap = 35
        feedline.y_pad_gap = 175
        feedline.arm_gap = feedline.feedline_gap 
        feedline.CPW = True
        feedline.Exclude_ground = True

        # CAD manager device info
        CAD_manager.feedline = feedline
        CAD_manager.dev_label = WaferName + '_V' + device_version + '_Wf' + WaferNumber + '_' + CAD_manager.chip
        CAD_manager.design_filename = WaferName + '_V' + device_version + '_Wf' + WaferNumber + '_' + CAD_manager.chip

        # Layer definition
        CAD_manager.ground_layer = ground_layer
        CAD_manager.pad_bottom_layer = feedline_layer
        CAD_manager.connection_wire_bottom_layer = feedline_layer
        CAD_manager.pad_top_layer = feedline_layer
        CAD_manager.connection_wire_top_layer = feedline_layer
        feedline.ground_layer = ground_layer
        feedline.feedline_layer = feedline_layer
        
        # gds and job file generation
        CAD_manager.generate_GDS()
        CAD_manager.populate_dose_matric()
        CAD_manager.generate_jobs(row=rows[i], column=columns[j])

## Job & Batch files generation

In [22]:
CAD_manager.batch_plan = batch_plan
CAD_manager.generate_batch_files()